# ROGII Wellbore Geology Prediction — 472nd-place solution

The full inference pipeline of the submission that scored **public RMSE 7.283 /
private 8.939 (472nd of 6,125 teams)**, cleaned up for publication. A markdown
cell explains each stage; the Japanese version of these notes is in
[`solution.ja.ipynb`](solution.ja.ipynb), and the method write-up is in the README.

The code is self-contained with this repository: every pretrained artifact the
notebook loads (GBDT stack, 15 GRU weights) is produced by the scripts under
[`train/`](train/), from the competition data alone.

**Pipeline** (the final cell runs everything in order):

1. **Core engine** — test features + pretrained GBDT stack ("GBDT-PP") + a
   128-seed particle filter matching the horizontal GR log against the
   typewell; writes the base prediction and a shared PF cache.
2. **Spatial-surface leg** — reconstructs the local stratigraphic surface from
   neighboring training wells.
3. **BiGRU residual refiner** — 15 pretrained models refine the
   particle-filter posterior; this becomes the prediction body.
4. **Duplicate-well recovery** and **contact override** — conservative,
   self-validating test-time post-processes.
5. **Runner** — base → GRU → per-well robust U-space projection → spatial
   blend → recoveries → `submission.csv`.


## Core engine — features, GBDT stack, and the 128-seed particle filter

Defines the self-contained base engine (executed by the runner as `main()`):

- **Feature builder** — rebuilds the per-row test features exactly as at training
  time (`train/build_features.py` is the same code): formation-plane KNN over
  neighboring training wells, a dense stratigraphic-surface imputer, multi-scale
  normalized cross-correlation of the horizontal GR log against the typewell,
  beam-search and light particle-filter summaries.
- **GBDT stack** — pretrained LightGBM/CatBoost boosters → Ridge stack on physics
  features → ridge blend → post-processing ensemble ("GBDT-PP"); trained by
  `train/train_gbdt.py`.
- **Particle filter** (`p128_*`) — per well, each particle advances a TVT
  hypothesis with the `dtvt ≈ −dz` structure plus dip dynamics, weighted by the
  agreement between observed GR and the typewell's GR at the hypothesized TVT.
  128 seeds × 500 particles; likelihood-weighted posterior mean, averaged across
  seeds. Results are cached on disk and the GRU leg reuses the same cache — the
  expensive filter runs once per well.

Base prediction = 0.20·GBDT-PP + 0.80·PF, followed by the U-space projection
(same code the runner applies again after the GRU leg). The base mostly serves as
a fallback — the GRU leg replaces it — but the PF posterior computed here is what
the GRU consumes.

In [ ]:
"""Core engine: features + GBDT stack + the 128-seed particle filter.

Inputs:
  - competition data /kaggle/input/.../{train,test}
  - the GBDT artifact dataset produced by train/train_gbdt.py (lgb*.txt, cb*.cbm,
    phys_ridge.pkl, ridge_blend.pkl, pp.json, meta.json)
Output:
  - submission.csv (test rows are rebuilt from /kaggle/input; the base prediction
    written here is refined by the later cells)

Flow of main():
  build test features with the same builder used at training time (spatial
  imputers rebuilt from the train wells) -> GBDT boosters -> physics Ridge stack
  -> ridge blend -> post-processing ensemble ("GBDT-PP")
  -> run the 128-seed particle filter on the test wells
  -> base prediction = 0.5 * GBDT-PP + 0.5 * PF.

Self-contained: the feature builder and the 128-seed PF live in this cell
(the PF uses the p128_/P128_ name prefix). Training code: train/ in the repo.
"""
from __future__ import annotations
import json, pickle, glob, time
from pathlib import Path
import numpy as np
import pandas as pd
from numba import njit
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed

# ============================================================================
#  Paths
# ============================================================================
try:
    _HERE = Path(__file__).resolve().parent          # script execution
except NameError:
    _HERE = Path.cwd()                                # notebook execution (no __file__)

def _find_comp_dir():
    for p in [Path("/kaggle/input/rogii-wellbore-geology-prediction"),
              Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
              Path("input"), _HERE.parents[1] / "input" if len(_HERE.parents) >= 2 else _HERE / "input"]:
        if (p / "train").exists(): return p
    raise FileNotFoundError("competition data not found")

def _find_artifact_dir():
    # local: train/artifacts/gbdt in the repo; Kaggle: attached artifact dataset
    for p in [_HERE / "train" / "artifacts" / "gbdt", _HERE / "artifacts"]:
        if (p / "lgb0.txt").exists(): return p
    for m in glob.glob("/kaggle/input/**/lgb0.txt", recursive=True):
        return Path(m).parent
    raise FileNotFoundError("artifact dataset (lgb0.txt) not found")

COMP = _find_comp_dir(); TRAIN = COMP / "train"; TEST = COMP / "test"
OUT_CSV = Path("submission.csv")
NCPU = 8

# ============================================================================
#  PART B: feature builder (identical to train/build_features.py)
# ============================================================================
FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10; DENSE_SPW = 60; DENSE_K = 20
BEAMS = [(10, 20.0, 144.0, 2, "cons"), (10, 8.0, 64.0, 2, "loose"), (8, 35.0, 220.0, 1, "vcons"),
         (10, 14.0, 90.0, 5, "sm5"), (20, 4.0, 36.0, 3, "vloose"), (12, 12.0, 100.0, 3, "mid"),
         (15, 25.0, 180.0, 2, "stiff")]
PF_N = 600; ANCC_N = 600
PF_MOM = 0.993; PF_VN = 0.005; PF_PN = 0.01
PF_GR_SIG_MIN = 10.; PF_GR_SIG_MAX = 60.; PF_GR_SIG_DEF = 30.
PF_RESAMP = 0.5; PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003; PF_GR_WIN = 5; PF_GR_WT = 0.3
ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001
ANCH_OFFS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], np.float32)
BEAM_OFFS = np.array([-40, -20, -10, -5, -3, 0, 3, 5, 10, 20, 40], np.float32)
SC_OFFS = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
PF_OFFS = SC_OFFS.copy()


@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i] * (1. - t) + grid[i + 1] * t

@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N + 1)
    for j in range(N): cum[j + 1] = cum[j] + w[j]
    u0 = np.random.uniform(0., 1. / N); np2 = np.empty(N); na = np.empty(N); ci = 0
    for j in range(N):
        u = u0 + j / N
        while ci < N - 1 and cum[ci + 1] < u: ci += 1
        np2[j] = pos[ci] + rp * np.random.randn(); na[j] = aux[ci] + rv * np.random.randn()
    return np2, na

@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]; cC[i], cC[mi] = cC[mi], cC[i]; cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

@njit(cache=True)
def _pf_ancc(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N, ALPHA, RN, PN, IS, RP, RR, RESAMP):
    pos = np.empty(N); rate = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ls + IS * np.random.randn(); rate[j] = ir + 0.01 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.
    for i in range(len(md_v)):
        dm = max(md_v[i] - pm, 1.)
        for j in range(N):
            rate[j] = ALPHA * rate[j] + RN * np.random.randn()
            pos[j] += rate[j] * dm + PN * np.random.randn()
            tvt_j = pos[j] - z_v[i]
            tvt_j = max(tvt_j, vmin - 50.); tvt_j = min(tvt_j, vmin + len(gg) * step + 50.)
            pos[j] = tvt_j + z_v[i]
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                eg = _interp1(gg, pos[j] - z_v[i], vmin, step); d = (gr_v[i] - eg) / gs
                lk = max(np.exp(-0.5 * d * d) if d * d < 600. else 0., 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, rate = _resamp(pos, rate, w, N, RP, RR)
            for j in range(N): w[j] = 1. / N
        tv = 0.
        for j in range(N): tv += w[j] * (pos[j] - z_v[i])
        pts[i] = tv; va = 0.
        for j in range(N): va += w[j] * (pos[j] - z_v[i] - tv) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]
    return pts, std_

@njit(cache=True)
def _pf_z(md_v, z_v, gr_v, gr_sm_v, gg_p, gg_s, vmin, step, gs, ip, iv, beta, icpt, zsig, N,
         MOM, VN, PN, GR_WT, RP, RV, RESAMP):
    pos = np.empty(N); vel = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ip + 0.5 * np.random.randn(); vel[j] = iv + 0.02 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.; pz = z_v[0] - 1.
    for i in range(len(md_v)):
        dm = max(md_v[i] - pm, 1.); dzd = (z_v[i] - pz) / dm; ve = beta * dzd + icpt
        for j in range(N):
            vel[j] = MOM * vel[j] + VN * np.random.randn(); pos[j] += vel[j] * dm + PN * np.random.randn()
            pos[j] = max(pos[j], vmin - 50.); pos[j] = min(pos[j], vmin + len(gg_p) * step + 50.)
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                ep = _interp1(gg_p, pos[j], vmin, step); dp = (gr_v[i] - ep) / gs
                lp = max(np.exp(-0.5 * dp * dp) if dp * dp < 600. else 0., 1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es = _interp1(gg_s, pos[j], vmin, step); ds = (gr_sm_v[i] - es) / (gs * 1.5)
                    lsm = max(np.exp(-0.5 * ds * ds) if ds * ds < 600. else 0., 1e-300)
                    lk = (1. - GR_WT) * lp + GR_WT * lsm
                else: lk = lp
                lk = max(lk, 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ws2 = 0.
        for j in range(N):
            dv = (vel[j] - ve) / max(zsig * 2., 0.005)
            lz = max(np.exp(-0.5 * dv * dv) if dv * dv < 600. else 0., 1e-300); w[j] *= lz; ws2 += w[j]
        if ws2 > 0.:
            for j in range(N): w[j] /= ws2
        else:
            for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, vel = _resamp(pos, vel, w, N, RP, RV)
            for j in range(N): w[j] = 1. / N
        wm = 0.
        for j in range(N): wm += w[j] * pos[j]
        pts[i] = wm; va = 0.
        for j in range(N): va += w[j] * (pos[j] - wm) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]; pz = z_v[i]
    return pts, std_


def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax + step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)

def _gr_sig(hw, tw_tvt, tw_gr):
    kn = hw[hw['TVT_input'].notna() & hw['GR'].notna()]
    if len(kn) < 20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values - np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)),
                         PF_GR_SIG_MIN, PF_GR_SIG_MAX))

def _nn(arr, v):
    i = int(np.searchsorted(arr, v, 'left'))
    if i >= len(arr): return len(arr) - 1
    if i > 0 and abs(arr[i - 1] - v) <= abs(arr[i] - v): return i - 1
    return i

def _smooth(vals, fb, r):
    s = pd.Series(vals, dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r * 2 + 1, center=True, min_periods=1).mean() if r > 0 else s).to_numpy(np.float32)

def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt); sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    return tw_tvt[_beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))].astype(np.float32)

def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr); kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    ls = float(kn['TVT_input'].iloc[-1] + kn['Z'].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail['TVT_input'].values); dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    pts, std = _pf_ancc(ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
                        ev['GR'].values.astype(np.float64), gg, gmin, gst, gs, ls, ir, N,
                        ANCC_ALPHA, ANCC_RN, ANCC_PN, ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)

def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    tw_s = pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1).mean().values.astype(np.float32)
    kna = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    dz_k = np.diff(kna['Z'].values); dvt = np.diff(kna['TVT_input'].values); dmd_k = np.diff(kna['MD'].values); m2 = dmd_k > 0
    if m2.sum() >= 10:
        vz = dz_k[m2] / dmd_k[m2]; vt = dvt[m2] / dmd_k[m2]
        A = np.column_stack([vz, np.ones_like(vz)]); c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt, zsig = float(c[0]), float(c[1]), max(float(np.std(vt - (c[0] * vz + c[1]))), 0.001)
    else: beta, icpt, zsig = -1., 0., 0.1
    t2 = kna.tail(20); dvt2 = np.diff(t2['TVT_input'].values); dmd2 = np.diff(t2['MD'].values); m3 = dmd2 > 0
    iv = float(np.median(dvt2[m3] / dmd2[m3])) if m3.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr); gs2, _, _ = _grid(tw_tvt, tw_s)
    gr_sm = hw['GR'].rolling(PF_GR_WIN, center=True, min_periods=1).mean()
    pts, std = _pf_z(ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
                     ev['GR'].values.astype(np.float64), gr_sm.loc[ev.index].values.astype(np.float64),
                     gg, gs2, gmin, gst, gs, float(kna['TVT_input'].iloc[-1]), iv, beta, icpt, zsig, N,
                     PF_MOM, PF_VN, PF_PN, PF_GR_WT, PF_ROUGH_P, PF_ROUGH_V, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)

def robust_slope(x, y, w=None):
    x = np.asarray(x, float); y = np.asarray(y, float); m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.std(x[m]) < 1e-6: return 0.
    return float(np.polyfit(x[m], y[m], 1)[0])

def affine_cal(kgr, tw_at_k, min_pts=20):
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        return 1., float(np.nanmean(kgr) - np.nanmean(tw_at_k)) if v.any() else 0.
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1); return float(a), float(b)

def seg_b_well(ktvt, kz, form_col):
    bv = ktvt + kz - form_col; n = len(bv); b_full = float(np.median(bv))
    b_late = float(np.median(bv[max(0, n - 50):])) if n >= 5 else b_full
    t1, t2 = n // 3, 2 * n // 3
    b_early = float(np.median(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.median(bv[t1:max(t1 + 1, t2)])) if t2 > t1 else b_full
    w = np.exp(0.02 * np.arange(n)); w /= w.sum(); b_wls = float(np.dot(w, bv))
    return b_full, b_early, b_mid, b_late, b_wls

def multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2 * hw + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32); M = len(sts)
        if M == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw, mode='edge')
        H = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]].astype(np.float32)
        Hn = (H - H.mean(1, keepdims=True)) / (H.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win; best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best] + hw, 0, nk - 1)].astype(np.float32), score))
    tvts = np.stack([o[0] for o in out], 1); scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3. * scores); sw /= sw.sum(1, keepdims=True) + 1e-9
    return out, (tvts * sw).sum(1).astype(np.float32)

class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            try: df = pd.read_csv(data_dir / f'{wid}__horizontal_well.csv', usecols=['X', 'Y'] + FORMATIONS).dropna()
            except Exception: continue
            if len(df) == 0: continue
            row = {'wid': wid, 'x': float(df['X'].median()), 'y': float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m'] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df['wid'])}
        xy = self.df[['x', 'y']].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df['x'].to_numpy(); self.ya = self.df['y'].to_numpy()
        self.fa = self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale; nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ordr = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w * xn; wy = w * yn
        A = np.zeros((len(q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1); A[:, 0, 1] = (wx * yn).sum(1); A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]; A[:, 1, 1] = (wy * yn).sum(1); A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]; A[:, 2, 1] = A[:, 1, 2]; A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
        rhs = np.stack([(wx[:, :, None] * fn).sum(1), (wy[:, :, None] * fn).sum(1), (w[:, :, None] * fn).sum(1)], 1)
        try: coef = np.linalg.solve(A, rhs)
        except Exception:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try: coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except Exception: pass
        Xq = xy_q[:, 0]; Yq = xy_q[:, 1]
        pred = (Xq[:, None] * coef[:, 0, :] + Yq[:, None] * coef[:, 1, :] + coef[:, 2, :]).astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)

class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, anccs, wids = [], [], [], []
        for wid in well_ids:
            try: df = pd.read_csv(data_dir / f'{wid}__horizontal_well.csv', usecols=['X', 'Y', 'ANCC']).dropna()
            except Exception: continue
            if len(df) == 0: continue
            ix = np.linspace(0, len(df) - 1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values); anccs.append(s['ANCC'].values); wids.extend([wid] * len(s))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32); self.wids = np.array(wids)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1., self.xy.std(0)); self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q / self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid: dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        ordr = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.)
        sw = w.sum(1); safe = np.where(sw < 1e-9, 1., sw); an = self.ancc[ik]
        ap = (an * w).sum(1) / safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((an - ap[:, None]) ** 2 * w).sum(1) / safe
        return (ap.astype(np.float32), np.sqrt(np.maximum(var, 0.)).astype(np.float32),
                np.where(vk, dk, np.inf).min(1).astype(np.float32))

_FI = None; _DI = None

def build_well(hw_path, tw_path, is_train):
    wid = Path(hw_path).stem.replace('__horizontal_well', '')
    try:
        hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path).sort_values('TVT')
    except Exception: return None
    if is_train and 'TVT' not in hw.columns: return None
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0 or len(kn) < 10: return None
    tw_tvt = tw['TVT'].to_numpy(np.float32); tw_gr = tw['GR'].to_numpy(np.float32)
    if len(tw_tvt) < 3: return None
    pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr)
    if len(pf_a) == 0: return None
    pf_z, std_z = run_pf_z(hw, tw_tvt, tw_gr)
    pf_use = pf_a.astype(np.float32); std_use = std_a.astype(np.float32)
    has_z = len(pf_z) == len(pf_a) and not np.any(np.isnan(pf_z))
    lk = kn.iloc[-1]; last_tvt = float(lk['TVT_input'])
    gr_full = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[ev.index[0]:].to_numpy(np.float32); kgr = gr_full.iloc[:len(kn)].to_numpy(np.float32)
    bpaths = {tag: beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for (bs, mc, es, r, tag) in BEAMS}
    beam_ref = (bpaths['cons'] + bpaths['sm5']) / 2.
    ktvt = kn['TVT_input'].to_numpy(np.float32)
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8, sc8s = sc_res[0]; sc15, sc15s = sc_res[1]; sc25, sc25s = sc_res[2]
    sc_cons = (sc8 + sc15 + sc25) / 3.
    sc_trust = float(np.clip(len(kn) / 200., 0., 0.6)); hyb_ref = (1 - sc_trust) * beam_ref + sc_trust * sc_ens
    tw_at_k = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32); a_cal, b_cal = affine_cal(kgr, tw_at_k)
    kmd = kn['MD'].to_numpy(np.float32); kz = kn['Z'].to_numpy(np.float32)
    pfx_rmse = float(np.sqrt(np.mean((kgr - tw_at_k) ** 2)))
    slp_all = robust_slope(kmd, ktvt); slp_50 = robust_slope(kmd[-50:], ktvt[-50:]); slp_z = robust_slope(kz, ktvt)
    swid = wid if is_train else None
    xy_ev = ev[['X', 'Y']].to_numpy(np.float64); xy_kn = kn[['X', 'Y']].to_numpy(np.float64)
    form_ev, knn_d = _FI.impute(xy_ev, self_wid=swid); form_kn, _ = _FI.impute(xy_kn, self_wid=swid)
    z_kn = kn['Z'].to_numpy(np.float32); z_ev = ev['Z'].to_numpy(np.float32)
    tvt_fs = {}; form_rmse = {}; form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev + form_ev[:, fi2] + b_full).astype(np.float32)
        tvt_fs[f'tvtF_{fn}'] = tvt_f; tvt_fs[f'tvtFw_{fn}'] = (-z_ev + form_ev[:, fi2] + b_wls).astype(np.float32)
        tvt_fs[f'tvtF50_{fn}'] = (-z_ev + form_ev[:, fi2] + b_late).astype(np.float32)
        tvt_fs[f'bw_{fn}'] = np.float32(b_full); tvt_fs[f'bww_{fn}'] = np.float32(b_wls); tvt_fs[f'bw50_{fn}'] = np.float32(b_late)
        tvt_fs[f'bw_early_{fn}'] = np.float32(b_early); tvt_fs[f'bw_mid_{fn}'] = np.float32(b_mid)
        form_rmse[fn] = float(np.sqrt(np.mean((ktvt - (-z_kn + form_kn[:, fi2] + b_full)) ** 2))); form_list.append(tvt_f)
    fs = np.stack(form_list, 1)
    form_mean_d = (fs.mean(1) - last_tvt).astype(np.float32); form_std_d = fs.std(1).astype(np.float32); form_rng_d = (fs.max(1) - fs.min(1)).astype(np.float32)
    d_ancc, d_std, d_dist = _DI.impute(xy_ev, self_wid=swid); d_kn, d_std_kn, _ = _DI.impute(xy_kn, self_wid=swid)
    _, b_de, b_dm, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn); b_d = float(np.median(ktvt + z_kn - d_kn))
    tvt_dense = (-z_ev + d_ancc + b_d).astype(np.float32); tvt_densew = (-z_ev + d_ancc + b_dw).astype(np.float32); tvt_dense50 = (-z_ev + d_ancc + b_dl).astype(np.float32)
    res_kn = ktvt + z_kn - d_kn; d_rmse = float(np.sqrt(np.mean(res_kn ** 2))); d_bias = float(np.mean(res_kn)); d_nb_std = float(np.mean(d_std_kn))
    all_sigs = [pf_use] + [p for p in bpaths.values()] + [sc8, sc15, sc25, sc_ens, tvt_fs['tvtF_ANCC'], tvt_dense]
    sig_mat = np.stack(all_sigs, 1); sig_std = sig_mat.std(1).astype(np.float32); sig_mean = (sig_mat.mean(1) - last_tvt).astype(np.float32)
    gr_s = pd.Series(gr_full.values); rolls = {}
    for w in [5, 21, 51, 101]:
        r = gr_s.rolling(w, center=True, min_periods=1)
        rolls[f'grm{w}'] = r.mean().iloc[ev.index].values.astype(np.float32); rolls[f'grs{w}'] = r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1, 5, 15, 30]:
        rolls[f'glag{lag}'] = gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32); rolls[f'glead{lag}'] = gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1 = gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32); gr_d2 = gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env = gr_s.rolling(21, center=True, min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg = np.sqrt(np.maximum((gr_s ** 2).rolling(21, center=True, min_periods=1).mean(), 0.)).iloc[ev.index].values.astype(np.float32)
    hmd = ev['MD'].to_numpy(np.float32); md_since = hmd - float(lk['MD'])
    slp_b_all = (last_tvt + slp_all * md_since).astype(np.float32); slp_b_50 = (last_tvt + slp_50 * md_since).astype(np.float32)
    mdd = hw['MD'].diff().replace(0, np.nan)
    dzdmd = (hw['Z'].diff() / mdd).iloc[ev.index].values.astype(np.float32); dxdmd = (hw['X'].diff() / mdd).iloc[ev.index].values.astype(np.float32); dydmd = (hw['Y'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    nh = len(ev); frac = (np.arange(nh) / max(nh - 1, 1)).astype(np.float32)
    def sc(v): return np.full(nh, np.float32(v), np.float32)
    feats = {
        'well': wid, 'id': [f'{wid}_{i}' for i in ev.index], 'last_known_tvt': sc(last_tvt),
        'pf_ancc': pf_use, 'pf_ancc_std': std_use, 'pf_ancc_delta': (pf_use - last_tvt).astype(np.float32),
        'pf_z': (pf_z.astype(np.float32) if has_z else sc(last_tvt)), 'pf_z_delta': ((pf_z - last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z': ((pf_use - pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f'beam_{t}_d': (p - np.float32(last_tvt)).astype(np.float32) for t, p in bpaths.items()},
        'beam_mean_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p - last_tvt) for p in bpaths.values()], 1), 1).astype(np.float32),
        'sc8_d': (sc8 - np.float32(last_tvt)).astype(np.float32), 'sc8_sc': sc8s,
        'sc15_d': (sc15 - np.float32(last_tvt)).astype(np.float32), 'sc15_sc': sc15s,
        'sc25_d': (sc25 - np.float32(last_tvt)).astype(np.float32), 'sc25_sc': sc25s,
        'sc_cons_d': (sc_cons - np.float32(last_tvt)).astype(np.float32), 'sc_ens_d': (sc_ens - np.float32(last_tvt)).astype(np.float32),
        'sc_trust': sc(sc_trust), 'hyb_d': (hyb_ref - np.float32(last_tvt)).astype(np.float32),
        'sig_std': sig_std, 'sig_mean_d': sig_mean, **tvt_fs,
        **{f'frm_rmse_{fn}': sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d': form_mean_d, 'form_std_d': form_std_d, 'form_rng_d': form_rng_d,
        'spatial_ancc_d': (form_ev[:, 0] - np.float32(np.interp(last_tvt, tw_tvt, tw_gr))), 'spatial_knn_dist': knn_d,
        'dense_ancc': d_ancc, 'dense_std': d_std, 'dense_dist': d_dist,
        'tvt_dense_d': (tvt_dense - last_tvt).astype(np.float32), 'tvt_densew_d': (tvt_densew - last_tvt).astype(np.float32), 'tvt_dense50_d': (tvt_dense50 - last_tvt).astype(np.float32),
        'dense_rmse': sc(d_rmse), 'dense_bias': sc(d_bias), 'dense_nb_std': sc(d_nb_std),
        'pf_vs_spatial': (pf_use - tvt_fs['tvtF_ANCC']).astype(np.float32), 'pf_vs_dense': (pf_use - tvt_dense).astype(np.float32),
        'spatial_vs_dense': (tvt_fs['tvtF_ANCC'] - tvt_dense).astype(np.float32), 'beam_vs_spatial': (bpaths['cons'] - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam': (sc_ens - bpaths['cons']).astype(np.float32), 'cal_a': sc(a_cal), 'cal_b': sc(b_cal),
        'pfx_rmse': sc(pfx_rmse), 'known_len': sc(len(kn)), 'eval_len': sc(nh),
        'slp_all': sc(slp_all), 'slp_50': sc(slp_50), 'slp_z': sc(slp_z),
        'slp_b_d_all': (slp_b_all - last_tvt).astype(np.float32), 'slp_b_d_50': (slp_b_50 - last_tvt).astype(np.float32),
        'ktvt_range': sc(float(np.ptp(ktvt))), 'ktvt_std': sc(float(ktvt.std())),
        'md_since': md_since, 'frac': frac, 'frac2': frac ** 2, 'sqrt_frac': np.sqrt(frac), 'z': z_ev,
        'dx': (ev['X'] - float(lk['X'])).to_numpy(np.float32), 'dy': (ev['Y'] - float(lk['Y'])).to_numpy(np.float32), 'dz': (z_ev - float(lk['Z'])).astype(np.float32),
        'dxy': np.sqrt((ev['X'] - float(lk['X'])) ** 2 + (ev['Y'] - float(lk['Y'])) ** 2).to_numpy(np.float32),
        'dzdmd': dzdmd, 'dxdmd': dxdmd, 'dydmd': dydmd,
        'gr': hgr, 'gr_d1': gr_d1, 'gr_d2': gr_d2, 'gr_env': gr_env, 'gr_nrg': gr_nrg,
        'gr_vs_tw_anc': hgr - np.float32(np.interp(last_tvt, tw_tvt, tw_gr)), 'gr_vs_slp_all': hgr - np.interp(slp_b_all, tw_tvt, tw_gr).astype(np.float32),
        **{f'tda{int(o)}': hgr - np.float32(np.interp(last_tvt + o, tw_tvt, tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}': hgr - np.interp(beam_ref + o, tw_tvt, tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}': hgr - np.interp(sc_ens + o, tw_tvt, tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f'tdpf{int(o)}': hgr - np.interp(pf_use + o, tw_tvt, tw_gr).astype(np.float32) for o in PF_OFFS},
        'tw_range': sc(float(np.ptp(tw_tvt))), 'tw_gr_mean': sc(float(tw_gr.mean())),
    }
    for k, v in rolls.items(): feats[k] = v
    return pd.DataFrame(feats)

def build_dataset(paths, is_train, n_jobs=NCPU):
    args = [(str(p), str(p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv'), is_train)
            for p in paths if (p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv').exists()]
    res = Parallel(n_jobs=n_jobs, prefer='threads')(delayed(build_well)(hp, tp, it) for hp, tp, it in args)
    parts = [r for r in res if r is not None]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


# ============================================================================
#  PART C: 128-seed particle filter (p128_/P128_ name prefix)
# ============================================================================
P128_N_PARTICLES = 500; P128_N_SEEDS = 128; P128_LIK_SCALE = 5.0; P128_INIT_SPREAD = 2.0
P128_MOM = 0.998; P128_VN = 0.002; P128_PN = 0.005; P128_RP = 0.1; P128_RR = 0.001; P128_RESAMP = 0.5
P128_GR_SIG_MIN = 10.0; P128_GR_SIG_MAX = 60.0
P128_N_EVAL_THRESH = 4840.0; P128_Z_SPAN_THRESH = (136.73, 185.51)
# known-zone GR debias before PF matching (fit-free, per-well; std-invariant so gs unchanged). "bias"|"none".
P128_GR_CALIB = "none"
# GR-trust knob. The PF works better when it trusts GR LESS (ill-conditioned matching). Scales gs (post-clip).
P128_GS_MULT = 1.0
P128_SELECTOR = {0: {"scale": 3.0, "beam_w": 0.00, "hold_w": 0.20}, 1: {"scale": 5.0, "beam_w": 0.00, "hold_w": 0.15},
                 2: {"scale": 3.0, "beam_w": 0.00, "hold_w": 0.15}, 3: {"scale": 3.0, "beam_w": 0.05, "hold_w": 0.05},
                 4: {"scale": 12.0, "beam_w": 0.30, "hold_w": 0.15}, 5: {"scale": 12.0, "beam_w": 0.30, "hold_w": 0.05}}
P128_FALLBACK = {"scale": 8.0, "beam_w": 0.00, "hold_w": 0.20}
P128_BEAM_CONFIGS = [(10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1), (10, 14.0, 90.0, 5),
                     (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3), (15, 25.0, 180.0, 2), (20, 30.0, 200.0, 2),
                     (15, 10.0, 80.0, 4), (25, 6.0, 50.0, 3), (10, 40.0, 300.0, 1), (12, 18.0, 120.0, 5),
                     (30, 8.0, 70.0, 2), (10, 50.0, 400.0, 0)]

@njit(cache=True)
def p128_interp(xs, xp, fp):
    out = np.empty(len(xs))
    for k in range(len(xs)):
        x = xs[k]
        if x <= xp[0]: out[k] = fp[0]
        elif x >= xp[-1]: out[k] = fp[-1]
        else:
            i = np.searchsorted(xp, x) - 1; t = (x - xp[i]) / (xp[i + 1] - xp[i]); out[k] = fp[i] + t * (fp[i + 1] - fp[i])
    return out

@njit(cache=True)
def p128_core(md_v, z_v, gr_v, tw_tvt, tw_gr, n_particles, seed, gs, last_tvt, last_z, last_md, ir,
              mom, vn, pn, rp, rr, resamp, init_spread, tmin, tmax):
    np.random.seed(seed)
    ls = last_tvt + last_z; pos = ls + init_spread * np.random.randn(n_particles)
    rate = ir + 0.01 * np.random.randn(n_particles); w = np.ones(n_particles) / n_particles
    res = np.empty(len(md_v)); log_lik = 0.0; prev_md = last_md
    for i in range(len(md_v)):
        dm = max(md_v[i] - prev_md, 1.0); rate = mom * rate + vn * np.random.randn(n_particles)
        pos = pos + rate * dm + pn * np.random.randn(n_particles)
        tvt_p = np.clip(pos - z_v[i], tmin - 100.0, tmax + 100.0); pos = tvt_p + z_v[i]
        if not np.isnan(gr_v[i]):
            eg = p128_interp(tvt_p, tw_tvt, tw_gr); d = (gr_v[i] - eg) / gs
            lk = np.exp(-0.5 * np.minimum(d ** 2, 600.0)); lk = np.maximum(lk, 1e-300)
            avg = np.dot(w, lk); log_lik += np.log(max(avg, 1e-300)); w = w * lk; ws = w.sum()
            w = w / ws if ws > 0.0 else np.ones(n_particles) / n_particles
        n_eff = 1.0 / np.dot(w, w)
        if n_eff < resamp * n_particles:
            cum = np.cumsum(w); u0 = np.random.uniform(0.0, 1.0 / n_particles)
            new_pos = np.empty(n_particles); new_rate = np.empty(n_particles)
            for j in range(n_particles):
                u = u0 + j / n_particles; idx = min(np.searchsorted(cum, u), n_particles - 1)
                new_pos[j] = pos[idx] + rp * np.random.randn(); new_rate[j] = rate[idx] + rr * np.random.randn()
            pos = new_pos; rate = new_rate; w = np.ones(n_particles) / n_particles
        res[i] = np.dot(w, pos) - z_v[i]; prev_md = md_v[i]
    return res, log_lik

def p128_smooth_gr(arr, window=11, poly=3):
    n = len(arr); w = window if window % 2 == 1 else window + 1; p = min(poly, w - 1)
    if n < w: w = max(3, n if n % 2 == 1 else n - 1); p = min(p, w - 1)
    if n < 3: return arr.copy()
    return savgol_filter(arr, w, p)

def p128_gr_sigma(hw, tw_tvt, tw_gr):
    kn = hw[hw["TVT_input"].notna()]
    if len(kn) < 20: return 30.0 * P128_GS_MULT
    tw_at_k = np.interp(kn["TVT_input"].values, tw_tvt, tw_gr)
    return float(np.clip(np.nanstd(kn["GR"].fillna(0).values - tw_at_k), P128_GR_SIG_MIN, P128_GR_SIG_MAX)) * P128_GS_MULT

def p128_gr_bias(hw, tw_tvt, tw_gr):
    """median(typewell_GR(TVT_known) - GR_known): shift removing the LWD vs typewell GR offset."""
    kn = hw[hw["TVT_input"].notna()]
    if len(kn) < 20: return 0.0
    gk = kn["GR"].interpolate(limit_direction="both").to_numpy(float)
    twk = np.interp(kn["TVT_input"].to_numpy(float), tw_tvt, tw_gr)
    ok = np.isfinite(gk) & np.isfinite(twk)
    if ok.sum() < 20: return 0.0
    return float(np.median(twk[ok] - gk[ok]))

def p128_run_pf(hw, tw_tvt, tw_gr, n_particles=P128_N_PARTICLES, seed=42):
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0: return hw["TVT_input"].values.astype(float).copy(), 0.0
    last_tvt = float(kn.iloc[-1]["TVT_input"]); last_z = float(kn.iloc[-1]["Z"]); last_md = float(kn.iloc[-1]["MD"])
    gs = p128_gr_sigma(hw, tw_tvt, tw_gr)
    tail = kn.tail(30); dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values); dm = np.diff(tail["MD"].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0
    gr_interp = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()); gr_v = gr_interp.values.astype(float)[ev.index]
    if P128_GR_CALIB == "bias":
        gr_v = gr_v + p128_gr_bias(hw, tw_tvt, tw_gr)
    tmin, tmax = float(tw_tvt.min()), float(tw_tvt.max())
    res, log_lik = p128_core(ev["MD"].values.astype(float), ev["Z"].values.astype(float), gr_v, tw_tvt, tw_gr,
                             n_particles, seed, gs, last_tvt, last_z, last_md, ir,
                             P128_MOM, P128_VN, P128_PN, P128_RP, P128_RR, P128_RESAMP, P128_INIT_SPREAD, tmin, tmax)
    out = hw["TVT_input"].values.astype(float).copy(); out[list(ev.index)] = res
    return out, log_lik

def p128_run_pf_ensemble(hw, tw_tvt, tw_gr, n_seeds=P128_N_SEEDS, scale=P128_LIK_SCALE):
    results = [p128_run_pf(hw, tw_tvt, tw_gr, seed=s) for s in range(n_seeds)]
    preds = np.stack([r[0] for r in results]); liks = np.array([r[1] for r in results])
    weights = np.exp((liks - liks.max()) / scale); weights /= weights.sum()
    return (weights[:, None] * preds).sum(0)

def p128_beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r):
    n = len(hgr)
    if r > 0 and n >= 3:
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1); win = max(win, 3); poly = min(2, win - 1); hgr = savgol_filter(hgr, win, poly)
    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int32); MC = np.array([2.0, 1.0, 0.0, 1.0, 2.0]) * mc
    nw = len(tw_tvt); n_mv = len(MOVES); si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
    bidx = np.full(bs, si, dtype=np.int32); bcost = np.zeros(bs, dtype=np.float64); result = np.empty(n)
    for i in range(n):
        gv = hgr[i]; new_idx = np.clip(bidx[:, None] + MOVES[None, :], 0, nw - 1)
        gr_err = np.zeros((bs, n_mv)) if np.isnan(gv) else (gv - tw_gr[new_idx]) ** 2 / es
        new_cost = bcost[:, None] + gr_err + MC[None, :]
        flat_idx = new_idx.reshape(-1); flat_cost = new_cost.reshape(-1)
        top_k = np.argpartition(flat_cost, bs)[:bs]; top_k = top_k[np.argsort(flat_cost[top_k])]
        bidx = flat_idx[top_k].astype(np.int32); bcost = flat_cost[top_k]; result[i] = tw_tvt[bidx[0]]
    return result

def p128_run_beam_ensemble(hw, tw_tvt, tw_gr, configs=P128_BEAM_CONFIGS):
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0: return hw["TVT_input"].values.astype(float).copy()
    last_tvt = float(kn.iloc[-1]["TVT_input"])
    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).values.astype(float); hgr = gr_all[list(ev.index)]
    results = [p128_beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for bs, mc, es, r in configs]
    out = hw["TVT_input"].values.astype(float).copy(); out[list(ev.index)] = np.stack(results, 0).mean(0)
    return out

def p128_apply_hold(pred, hw, hold_w):
    if hold_w <= 0 or not hw["TVT_input"].notna().any(): return pred
    last_tvt = float(hw["TVT_input"].dropna().iloc[-1]); anchor_md = float(hw.loc[hw["TVT_input"].notna(), "MD"].iloc[-1])
    ev = hw[hw["TVT_input"].isna()]; ev_md = ev["MD"].values.astype(float); ev_idx = list(ev.index); md_range = ev_md.max() - anchor_md
    out = pred.copy()
    if md_range <= 0: out[ev_idx] = (1.0 - hold_w) * pred[ev_idx] + hold_w * last_tvt; return out
    depth_frac = np.clip((ev_md - anchor_md) / md_range, 0.0, 1.0); hw_eff = hold_w * np.maximum(0.0, 1.0 - depth_frac)
    out[ev_idx] = (1.0 - hw_eff) * pred[ev_idx] + hw_eff * last_tvt; return out

def p128_classify(hw_ev):
    n_eval = len(hw_ev); z_span = float(hw_ev["Z"].max() - hw_ev["Z"].min()) if len(hw_ev) > 0 else 0.0
    n_bin = int(n_eval > P128_N_EVAL_THRESH)
    z_bin = (0 if z_span <= P128_Z_SPAN_THRESH[0] else 1 if z_span <= P128_Z_SPAN_THRESH[1] else 2)
    return P128_SELECTOR.get(n_bin + 2 * z_bin, P128_FALLBACK)

def p128_predict(hw, tw_tvt, tw_gr):
    ev = hw[hw["TVT_input"].isna()]; cfg = p128_classify(ev)
    pred = p128_run_pf_ensemble(hw, tw_tvt, tw_gr, scale=cfg["scale"])
    if cfg["beam_w"] > 0 and len(ev) > 0:
        bm = p128_run_beam_ensemble(hw, tw_tvt, tw_gr); pred = (1.0 - cfg["beam_w"]) * pred + cfg["beam_w"] * bm
    if cfg["hold_w"] > 0: pred = p128_apply_hold(pred, hw, cfg["hold_w"])
    return pred


# ============================================================================
#  PART D: inference -> blend -> submission
# ============================================================================
PHYS_DIRECT = ['pf_ancc_delta', 'pf_z_delta', 'hyb_d', 'beam_cons_d', 'beam_loose_d', 'beam_vcons_d',
               'beam_sm5_d', 'beam_vloose_d', 'beam_mid_d', 'beam_stiff_d', 'beam_mean_d', 'beam_med_d',
               'sc_cons_d', 'sc_ens_d', 'tvt_dense_d', 'tvt_densew_d', 'tvt_dense50_d', 'form_mean_d',
               'slp_b_d_all', 'slp_b_d_50']

def _clean(v): return np.nan_to_num(np.asarray(v, np.float32), nan=0., posinf=0., neginf=0.).astype(np.float32)

def physics_cols(df, keep_cols):
    base = df['last_known_tvt'].to_numpy(np.float32); cand = {}
    for c in [c for c in keep_cols]:
        if c.endswith('_delta') and c[:-6] in ('pf_ancc', 'pf_z') or c[:-6].startswith(('tvtF_', 'tvtFw_', 'tvtF50_')):
            src = c[:-6]
            cand[c] = _clean(df[src].to_numpy(np.float32) - base) if src in df.columns else np.zeros(len(df), np.float32)
        else:
            cand[c] = _clean(df[c].to_numpy(np.float32)) if c in df.columns else np.zeros(len(df), np.float32)
    return pd.DataFrame(cand)[keep_cols].astype(np.float32)

def sg_smooth_groups(well, vals, sg_w=17, sg_p=3):
    out = vals.copy(); df = pd.DataFrame({"well": well, "v": vals})
    for _, g in df.groupby("well", sort=False):
        v = g["v"].values; n = len(v); wl = min(sg_w, n)
        if wl % 2 == 0: wl -= 1
        if wl >= sg_p + 2: out[g.index.values] = savgol_filter(v, wl, sg_p)
    return out

def apply_pp(md_since, model_delta, phys_delta, alpha, tau, w_pf):
    d = model_delta * (1 - w_pf) + phys_delta * w_pf
    if tau: d = d * (1. - np.exp(-np.maximum(md_since, 0.) / tau))
    return d * alpha


# ============================================================================
#  Post-processing recipe for the base prediction (identical code path to the
#  local by-well CV harness, so CV and submission stay byte-consistent)
# ============================================================================
RECIPE = {"name": "gbdt020_pf080_proj_d4cauchy",
          "w": {"gbdt": 0.20, "pf": 0.80},
          "project": {"degree": 4, "loss": "cauchy"}}


def ph_blend(D, w):
    return w.get("gbdt", 0.0) * D["gbdt"] + w.get("pf", 0.0) * D["pf"]


def ph_robust_polyfit(t, U, degree, loss, iters=5):
    A = np.vander(t, degree + 1); w = np.ones_like(U); coef = None
    for _ in range(iters):
        sw = np.sqrt(w)
        try:
            coef, *_ = np.linalg.lstsq(A * sw[:, None], U * sw, rcond=None)
        except Exception:
            return None
        r = U - A @ coef
        s = 1.4826 * np.median(np.abs(r - np.median(r))) + 1e-9
        if loss == "huber":
            c = 1.345 * s; a = np.abs(r); w = np.where(a <= c, 1.0, c / np.maximum(a, 1e-9))
        else:
            c = 2.385 * s; w = 1.0 / (1.0 + (r / c) ** 2)
    return coef, A


def ph_project_uspace(p, D, degree=4, loss="cauchy", min_pts=5, lam=1.0):
    out = p.astype(np.float64).copy(); z = D["z"]; mds = D["md_since"]
    need = max(min_pts, degree + 2)
    for _, idx in D["groups"]:
        if len(idx) < need: continue
        t = mds[idx]; rng = t.max() - t.min()
        if rng < 1e-9: continue
        tn = (t - t.min()) / rng; U = p[idx] + z[idx]
        res = ph_robust_polyfit(tn, U, degree, loss)
        if res is None: continue
        coef, A = res; proj = (A @ coef) - z[idx]
        out[idx] = lam * proj + (1.0 - lam) * p[idx]
    return out


def build_groups(well):
    return [(w, g.to_numpy()) for w, g in
            pd.Series(np.arange(len(well))).groupby(np.asarray(well), sort=False)]


def apply_recipe(cfg, D):
    p = ph_blend(D, cfg["w"]).astype(np.float64)
    if cfg.get("project"):
        p = ph_project_uspace(p, D, **cfg["project"])
    return p


def main():
    global _FI, _DI
    import lightgbm as lgb
    t0 = time.time()
    ART = _find_artifact_dir()
    meta = json.load(open(ART / "meta.json")); feat_cols = meta["feat_cols"]; BLEND_W = meta["blend_w"]
    print(f"COMP={COMP}  ART={ART}  blend_w={BLEND_W}")

    train_wids = [p.stem.replace('__horizontal_well', '') for p in sorted(TRAIN.glob('*__horizontal_well.csv'))]
    _FI = FormationPlaneKNN(train_wids, TRAIN); _DI = DenseANCCImputer(train_wids, TRAIN)
    print(f"FI/DI built on {len(train_wids)} train wells ({time.time()-t0:.0f}s)")

    test_paths = sorted(TEST.glob('*__horizontal_well.csv'))
    test_df = build_dataset(test_paths, is_train=False)
    print(f"test features: {test_df.shape}  ({time.time()-t0:.0f}s)")
    test_X = test_df[feat_cols].astype(np.float32)
    base = test_df['last_known_tvt'].values.astype(np.float32)

    # GBDT base (LightGBM boosters + CatBoost .cbm)
    preds = {}
    for k in meta["lgb_keys"]:
        bst = lgb.Booster(model_file=str(ART / f"{k}.txt")); preds[k] = bst.predict(test_X).astype(np.float32)
    if meta.get("cb_keys"):
        from catboost import CatBoostRegressor
        for k in meta["cb_keys"]:
            m = CatBoostRegressor(); m.load_model(str(ART / f"{k}.cbm")); preds[k] = m.predict(test_X).astype(np.float32)
    # physics stacks
    pr = pickle.load(open(ART / "phys_ridge.pkl", "rb")); keep = pr["keep_cols"]
    Xp = physics_cols(test_df, keep).to_numpy(np.float32)
    for name, m in pr["models"].items():
        preds[name] = ((Xp - m["mu"]) / m["sd"] @ m["coef"] + m["intercept"]).astype(np.float32)
    # ridge blend
    rb = pickle.load(open(ART / "ridge_blend.pkl", "rb"))
    O = np.column_stack([preds[k] for k in rb["keys"]]).astype(np.float64)
    ridge_delta = (O @ rb["coef"] + rb["intercept"]).astype(np.float32)
    # PP ensemble
    pp = json.load(open(ART / "pp.json")); pf_single = (test_df['pf_ancc'].values - base).astype(np.float32)
    md_since = test_df['md_since'].values.astype(np.float32); well = test_df['well'].values
    gbdt_pp = np.zeros(len(test_df), np.float32)
    for wt, p in zip(pp["weights"], pp["params"]):
        d = apply_pp(md_since, ridge_delta, pf_single, p['alpha'], p['tau'], p['w_pf'])
        gbdt_pp += wt * sg_smooth_groups(well, (base + d).astype(np.float32), int(p['sg_w']))
    print(f"GBDT-PP done ({time.time()-t0:.0f}s)")

    # 128-seed PF on the test wells (parallel per well)
    def _pf_one(hp):
        wid = hp.stem.replace('__horizontal_well', ''); tp = TEST / f"{wid}__typewell.csv"
        if not tp.exists(): return {}
        hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
        ev = hw[hw["TVT_input"].isna()]
        if len(ev) == 0: return {}
        tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].fillna(tw["GR"].mean()).to_numpy(float)
        pred = p128_predict(hw, tw_tvt, tw_gr)
        return {f"{wid}_{i}": float(pred[i]) for i in ev.index}
    maps = Parallel(n_jobs=NCPU, prefer="processes")(delayed(_pf_one)(hp) for hp in test_paths)
    pf_map = {}
    for mp in maps: pf_map.update(mp)
    pf128 = test_df['id'].map(pf_map).astype(np.float32).values
    print(f"PF128 done ({time.time()-t0:.0f}s)")

    # apply the post-processing recipe (same code path as the CV harness)
    Dtest = {
        "gbdt": gbdt_pp.astype(np.float64), "pf": pf128.astype(np.float64),
        "z": test_df["z"].to_numpy(np.float64), "md_since": test_df["md_since"].to_numpy(np.float64),
        "well": test_df["well"].to_numpy(), "groups": build_groups(test_df["well"].to_numpy()),
    }
    print(f"RECIPE={RECIPE['name']}")
    final = apply_recipe(RECIPE, Dtest).astype(np.float32)
    pred_df = pd.DataFrame({"id": test_df['id'].values, "tvt": final})

    sample = pd.read_csv(COMP / "sample_submission.csv")
    sub = sample[["id"]].merge(pred_df, on="id", how="left")
    fill = float(base.mean()) if len(base) else 0.0
    sub["tvt"] = sub["tvt"].fillna(fill)
    sub[["id", "tvt"]].to_csv(OUT_CSV, index=False)
    print(f"saved {OUT_CSV}: {len(sub)} rows, NaN={sub['tvt'].isna().sum()}  total {time.time()-t0:.0f}s")




# ===== PF128 shared on-disk cache (the base pass writes, the GRU leg reads) =====
import hashlib as _pfhl, os as _pfos
PF_CACHE_DIR = "/tmp/pfcache108"
_pfos.makedirs(PF_CACHE_DIR, exist_ok=True)
_p128_run_pf_raw = p128_run_pf

def _pf_cache_key(hw, n_particles, seed):
    h = _pfhl.sha1()
    h.update(hw["MD"].to_numpy(float).tobytes())
    h.update(hw["TVT_input"].to_numpy(float).tobytes())
    h.update(hw["GR"].to_numpy(float).tobytes())
    h.update(("p{}_s{}".format(int(n_particles), int(seed))).encode())
    return h.hexdigest()

def p128_run_pf(hw, tw_tvt, tw_gr, n_particles=P128_N_PARTICLES, seed=42):
    fp = _pfos.path.join(PF_CACHE_DIR, _pf_cache_key(hw, n_particles, seed) + ".npz")
    if _pfos.path.exists(fp):
        try:
            with np.load(fp) as z:
                return z["out"].astype(np.float64), float(z["lik"])
        except Exception:
            pass
    out, lik = _p128_run_pf_raw(hw, tw_tvt, tw_gr, n_particles=n_particles, seed=seed)
    try:
        tmp = fp[:-4] + ".w{}.npz".format(_pfos.getpid())
        np.savez(tmp, out=out.astype(np.float32), lik=np.float64(lik))
        _pfos.replace(tmp, fp)
    except Exception:
        pass
    return out, lik

## Spatial-surface leg

Defines `run_spatial_leg`, built on the structural identity
`TVT = ANCC(X, Y) − Z + b_well`:

- the ANCC stratigraphic surface is reconstructed by inverse-distance weighting
  (k=20 neighbors, power 2) over row-level point clouds of neighboring training
  wells;
- the per-well offset `b_well` comes from the known prefix
  (median of `TVT_input + Z − ANCC_hat`; leak-free).

Guards restrict it to wells where the reconstruction is trustworthy: ≥ 30
known-prefix rows, a well-conditioned neighborhood, prefix residual ≤ 20 ft.
The runner uses it twice — as a feature feed for the GRU (guards off) and as the
final W=0.30 outer blend (guards on). The same module is `train/spatial_leg.py`.

In [ ]:
"""Spatial-surface leg: transfer the geology from neighboring training wells.

Structural identity: TVT = ANCC(X, Y) - Z + b_well
  ANCC   = the ANCC stratigraphic surface, reconstructed by IDW over the
           row-level dense point clouds of neighboring training wells
  b_well = per-well offset fixed from the known prefix as
           median(TVT_input + Z - ANCC_hat)  (leak-free)

Self-contained (numpy/scipy/pandas only); all settings live in CONFIG.
Used both embedded in the submission kernel and imported by train/pf_dump.py
(the LOO spatial feature fed to the GRU during training).
"""
from __future__ import annotations
import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree


CONFIG = dict(
    SPW=60,               # samples per well for the point cloud (0 = all rows)
    K=20,                 # IDW neighbor count
    POWER=2.0,            # IDW distance power
    VARIANT="idw_trend",  # selected variant (global plane trend + IDW residual)
    ANISO_LAMBDA=1.0,     # anisotropy compression (idw_aniso only, unused here)
    ANISO_MODE="pca",     # "pca" | "heading"
    B_CAL="late",         # b calibration: "full" | "late"
    # -- guard thresholds (calibrated on the by-well CV curves) ---------------
    # NN_DIST_MAX: nearest-neighbor distance (scaled units) past which the
    #   surface reconstruction degrades sharply -> skip such wells.
    # PREFIX_RESID_MAX: self-validation RMSE of the reconstruction against the
    #   known prefix; wells above it are not trustworthy for this leg.
    NN_DIST_MAX=0.010,    # nearest-neighbor distance threshold (scaled)
    PREFIX_MIN=30,        # minimum known-prefix rows (below -> skip)
    PREFIX_RESID_MAX=20.0,  # prefix RMSE threshold (ft)
)


# ---------------------------------------------------------------- DenseCloud
class DenseCloud:
    """Row-level (X, Y, ANCC) point cloud over the training wells.
    Thinned per well by linspace (spw); coordinates scaled by their std.
    """

    def __init__(self):
        self.xy: np.ndarray = np.zeros((0, 2), np.float32)
        self.ancc: np.ndarray = np.zeros(0, np.float32)
        self.wids: np.ndarray = np.array([], dtype=object)
        self.scale: np.ndarray = np.ones(2, np.float64)
        self.heading: dict[str, float] = {}  # wid -> mean heading (rad)

    @classmethod
    def build(cls, train_dir: Path, spw: int = 60) -> "DenseCloud":
        """Build the point cloud from every __horizontal_well.csv in train_dir."""
        c = cls()
        xs, ys, anccs, wids = [], [], [], []
        headings: dict[str, float] = {}
        for p in sorted(train_dir.glob("*__horizontal_well.csv")):
            wid = p.stem.replace("__horizontal_well", "")
            try:
                df = pd.read_csv(p, usecols=["X", "Y", "ANCC"]).dropna()
            except Exception:
                continue
            if len(df) < 2:
                continue
            n = len(df)
            if spw > 0 and n > spw:
                ix = np.linspace(0, n - 1, spw, dtype=int)
            else:
                ix = np.arange(n)
            s = df.iloc[ix]
            xs.append(s["X"].values.astype(np.float32))
            ys.append(s["Y"].values.astype(np.float32))
            anccs.append(s["ANCC"].values.astype(np.float32))
            wids.extend([wid] * len(ix))
            # per-well mean heading (mean of 2*theta)
            dx = np.diff(df["X"].values)
            dy = np.diff(df["Y"].values)
            angles = np.arctan2(dy, dx)
            headings[wid] = float(np.angle(np.mean(np.exp(2j * angles)))) / 2
        c.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)]).astype(np.float64)
        c.ancc = np.concatenate(anccs).astype(np.float32)
        c.wids = np.array(wids, dtype=object)
        raw_std = c.xy.std(0)
        c.scale = np.where(raw_std < 1e-3, 1.0, raw_std)
        c.heading = headings
        return c

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        head_wids = np.array(list(self.heading.keys()), dtype=object)
        head_vals = np.array(list(self.heading.values()), dtype=np.float32)
        np.savez_compressed(
            str(path),
            xy=self.xy.astype(np.float32),
            ancc=self.ancc,
            wids=self.wids,
            scale=self.scale,
            head_wids=head_wids,
            head_vals=head_vals,
        )

    @classmethod
    def load(cls, path: Path) -> "DenseCloud":
        d = np.load(str(path), allow_pickle=True)
        c = cls()
        c.xy = d["xy"].astype(np.float64)
        c.ancc = d["ancc"].astype(np.float32)
        c.wids = d["wids"]
        c.scale = d["scale"].astype(np.float64)
        c.heading = dict(zip(d["head_wids"].tolist(), d["head_vals"].tolist()))
        return c


# ---------------------------------------------------------------- predictor classes
class _IdwPredictor:
    """Plain IDW: isotropic scaled distance; the self well's points are excluded."""

    def __init__(self, cloud: DenseCloud, k: int, power: float):
        self.cloud = cloud
        self.k = k
        self.power = power
        xy_sc = cloud.xy / cloud.scale
        self.tree = cKDTree(xy_sc)

    def predict(self, self_wid: str, xy_q: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """
        xy_q: (N,2) raw coordinates
        returns: (ancc_hat [N], nn_dist_scaled [N])
        """
        xy_q = np.atleast_2d(xy_q).astype(np.float64)
        q_sc = xy_q / self.cloud.scale
        nfetch = min(self.k + 500, len(self.cloud.ancc))
        dist, idx = self.tree.query(q_sc, k=nfetch, workers=-1)
        # drop the query well's own points
        mask_self = self.cloud.wids[idx] == self_wid
        dist = np.where(mask_self, np.inf, dist)
        k_use = min(self.k, nfetch)
        ord_ = np.argpartition(dist, min(k_use - 1, nfetch - 1), axis=1)[:, :k_use]
        dk = np.take_along_axis(dist, ord_, axis=1)
        ik = np.take_along_axis(idx, ord_, axis=1)
        vk = np.isfinite(dk)
        w = np.where(vk, 1.0 / (dk ** self.power + 1e-9), 0.0)
        an = self.cloud.ancc[ik].astype(np.float64)
        sw = w.sum(1)
        safe = np.where(sw < 1e-12, 1.0, sw)
        ancc_hat = (w * an).sum(1) / safe
        ancc_hat = np.where(sw < 1e-12, float(self.cloud.ancc.mean()), ancc_hat)
        nn_dist = np.where(vk, dk, np.inf).min(1)
        return ancc_hat.astype(np.float32), nn_dist.astype(np.float32)


class _IdwAnisoPredictor:
    """Anisotropic IDW: rotate to the principal heading and compress the
    orthogonal axis by lambda."""

    def __init__(self, cloud: DenseCloud, k: int, power: float,
                 lam: float = 1.0, mode: str = "pca"):
        self.cloud = cloud
        self.k = k
        self.power = power
        # transform matrix
        xy_sc = cloud.xy / cloud.scale
        if mode == "pca":
            mu = xy_sc.mean(0)
            c = xy_sc - mu
            cov = (c.T @ c) / max(len(c) - 1, 1)
            evals, evecs = np.linalg.eigh(cov)
            order = np.argsort(evals)[::-1]
            self.R = evecs[:, order]  # (2,2)
            self.mu = mu
        else:  # heading
            angles = np.array(list(cloud.heading.values()))
            mean_angle = float(np.angle(np.mean(np.exp(2j * angles)))) / 2
            self.R = np.array([[np.cos(mean_angle), -np.sin(mean_angle)],
                                [np.sin(mean_angle),  np.cos(mean_angle)]])
            self.mu = xy_sc.mean(0)
        # compression: principal axis kept, orthogonal axis scaled by lam
        self.compress = np.array([1.0, lam])
        txy = ((xy_sc - self.mu) @ self.R) * self.compress
        self.tree = cKDTree(txy)
        self.txy = txy

    def _transform(self, xy_q: np.ndarray) -> np.ndarray:
        q_sc = xy_q / self.cloud.scale
        return ((q_sc - self.mu) @ self.R) * self.compress

    def predict(self, self_wid: str, xy_q: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        xy_q = np.atleast_2d(xy_q).astype(np.float64)
        tq = self._transform(xy_q)
        nfetch = min(self.k + 500, len(self.cloud.ancc))
        dist, idx = self.tree.query(tq, k=nfetch, workers=-1)
        mask_self = self.cloud.wids[idx] == self_wid
        dist = np.where(mask_self, np.inf, dist)
        k_use = min(self.k, nfetch)
        ord_ = np.argpartition(dist, min(k_use - 1, nfetch - 1), axis=1)[:, :k_use]
        dk = np.take_along_axis(dist, ord_, axis=1)
        ik = np.take_along_axis(idx, ord_, axis=1)
        vk = np.isfinite(dk)
        w = np.where(vk, 1.0 / (dk ** self.power + 1e-9), 0.0)
        an = self.cloud.ancc[ik].astype(np.float64)
        sw = w.sum(1)
        safe = np.where(sw < 1e-12, 1.0, sw)
        ancc_hat = (w * an).sum(1) / safe
        ancc_hat = np.where(sw < 1e-12, float(self.cloud.ancc.mean()), ancc_hat)
        nn_dist = np.where(vk, dk, np.inf).min(1)
        return ancc_hat.astype(np.float32), nn_dist.astype(np.float32)


class _IdwTrendPredictor:
    """Subtract a global plane trend, then IDW the residuals."""

    def __init__(self, cloud: DenseCloud, k: int, power: float):
        self.cloud = cloud
        self.k = k
        self.power = power
        xy_sc = cloud.xy / cloud.scale
        self.xy_sc = xy_sc
        # global plane fit (all points)
        ok = np.isfinite(cloud.ancc)
        mu = xy_sc[ok].mean(0)
        sc = np.where(xy_sc[ok].std(0) < 1e-9, 1.0, xy_sc[ok].std(0))
        A = np.column_stack([np.ones(ok.sum()), (xy_sc[ok] - mu) / sc])
        beta, *_ = np.linalg.lstsq(A, cloud.ancc[ok].astype(np.float64), rcond=None)
        self.mu = mu
        self.sc = sc
        self.beta = beta
        self.resid = cloud.ancc.astype(np.float64) - self._trend(xy_sc)
        self.tree = cKDTree(xy_sc)

    def _trend(self, xy_sc: np.ndarray) -> np.ndarray:
        A = np.column_stack([np.ones(len(xy_sc)), (xy_sc - self.mu) / self.sc])
        return A @ self.beta

    def predict(self, self_wid: str, xy_q: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        xy_q = np.atleast_2d(xy_q).astype(np.float64)
        q_sc = xy_q / self.cloud.scale
        trend_q = self._trend(q_sc)
        nfetch = min(self.k + 500, len(self.cloud.ancc))
        dist, idx = self.tree.query(q_sc, k=nfetch, workers=-1)
        mask_self = self.cloud.wids[idx] == self_wid
        dist = np.where(mask_self, np.inf, dist)
        k_use = min(self.k, nfetch)
        ord_ = np.argpartition(dist, min(k_use - 1, nfetch - 1), axis=1)[:, :k_use]
        dk = np.take_along_axis(dist, ord_, axis=1)
        ik = np.take_along_axis(idx, ord_, axis=1)
        vk = np.isfinite(dk)
        w = np.where(vk, 1.0 / (dk ** self.power + 1e-9), 0.0)
        rn = np.where(vk, self.resid[ik], 0.0)
        sw = w.sum(1)
        safe = np.where(sw < 1e-12, 1.0, sw)
        r_hat = (w * rn).sum(1) / safe
        r_hat = np.where(sw < 1e-12, 0.0, r_hat)
        ancc_hat = trend_q + r_hat
        nn_dist = np.where(vk, dk, np.inf).min(1)
        return ancc_hat.astype(np.float32), nn_dist.astype(np.float32)


class _WlsLocalPredictor:
    """Local IRLS (Huber) plane: weighted local plane fit on the k neighbors."""

    def __init__(self, cloud: DenseCloud, k: int, power: float):
        self.cloud = cloud
        self.k = k
        self.power = power
        xy_sc = cloud.xy / cloud.scale
        self.xy_sc = xy_sc
        self.tree = cKDTree(xy_sc)

    def predict(self, self_wid: str, xy_q: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        xy_q = np.atleast_2d(xy_q).astype(np.float64)
        q_sc = xy_q / self.cloud.scale
        nfetch = min(self.k + 500, len(self.cloud.ancc))
        dist, idx = self.tree.query(q_sc, k=nfetch, workers=-1)
        mask_self = self.cloud.wids[idx] == self_wid
        dist = np.where(mask_self, np.inf, dist)
        k_use = min(self.k, nfetch)
        ord_ = np.argpartition(dist, min(k_use - 1, nfetch - 1), axis=1)[:, :k_use]
        dk = np.take_along_axis(dist, ord_, axis=1)
        ik = np.take_along_axis(idx, ord_, axis=1)
        ancc_hat = np.zeros(len(xy_q), np.float64)
        nn_dist = np.zeros(len(xy_q), np.float32)
        for i in range(len(xy_q)):
            vk_i = np.isfinite(dk[i])
            nn_dist[i] = float(dk[i][vk_i].min()) if vk_i.any() else np.inf
            if not vk_i.any():
                ancc_hat[i] = float(self.cloud.ancc.mean())
                continue
            nb_xy = self.xy_sc[ik[i][vk_i]]  # (K,2)
            nb_an = self.cloud.ancc[ik[i][vk_i]].astype(np.float64)
            nb_d = dk[i][vk_i]
            qc = q_sc[i]
            # center on the query point
            dx = nb_xy[:, 0] - qc[0]
            dy = nb_xy[:, 1] - qc[1]
            A = np.column_stack([np.ones(len(dx)), dx, dy])
            w_init = 1.0 / (nb_d ** self.power + 1e-9)
            # IRLS (Huber, 5 iter)
            w = w_init.copy()
            for _ in range(5):
                W = np.diag(w)
                try:
                    beta = np.linalg.solve(A.T @ W @ A + 1e-6 * np.eye(3), A.T @ W @ nb_an)
                except np.linalg.LinAlgError:
                    beta = np.array([float(np.nanmean(nb_an)), 0.0, 0.0])
                    break
                resid = nb_an - A @ beta
                sigma = max(np.median(np.abs(resid)) * 1.4826, 1e-6)
                r_sc = resid / sigma
                delta = 1.345
                huber_w = np.where(np.abs(r_sc) <= delta, 1.0, delta / np.abs(r_sc))
                w = w_init * huber_w
            ancc_hat[i] = beta[0]  # b0 = intercept at the centered origin = value at the query point
        return ancc_hat.astype(np.float32), nn_dist


def make_predictor(cloud: DenseCloud, variant: str = "idw", **kw):
    """Instantiate the predictor for the requested variant."""
    k = int(kw.get("k", CONFIG["K"]))
    power = float(kw.get("power", CONFIG["POWER"]))
    if variant == "idw":
        return _IdwPredictor(cloud, k, power)
    elif variant == "idw_aniso":
        lam = float(kw.get("lam", CONFIG["ANISO_LAMBDA"]))
        mode = str(kw.get("mode", CONFIG["ANISO_MODE"]))
        return _IdwAnisoPredictor(cloud, k, power, lam=lam, mode=mode)
    elif variant == "idw_trend":
        return _IdwTrendPredictor(cloud, k, power)
    elif variant == "wls_local":
        return _WlsLocalPredictor(cloud, k, power)
    else:
        raise ValueError(f"Unknown variant: {variant}")


# ---------------------------------------------------------------- b calibration
def _robust_slope(x: np.ndarray, y: np.ndarray) -> float:
    """Drift slope of b vs measured depth."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3 or np.std(x[m]) < 1e-6:
        return 0.0
    return float(np.polyfit(x[m], y[m], 1)[0])


def spatial_tvt_for_well(
    hw_df: pd.DataFrame,
    predictor,
    self_wid: str,
) -> dict:
    """
    One well: dense ANCC IDW -> three b calibrations -> three TVT_sp series
    plus diagnostics.

    Returns a dict with:
      kn_idx / ev_idx        prefix and eval row indices
      ancc_hat_all           ANCC_hat on all rows (raw coords)
      nn_dist_all            nearest-neighbor distance on all rows (scaled)
      b_full / b_late / b_slope / b_PS / md_PS
      tvt_bfull / tvt_blate / tvt_drift   eval-row TVT_sp series
      prefix_resid_rmse      prefix self-validation RMSE (TVT_sp vs TVT_input)
      n_kn / n_ev / nn_dist_ev / md_since
    """
    kn = hw_df[hw_df["TVT_input"].notna()]
    ev = hw_df[hw_df["TVT_input"].isna()]
    if len(ev) == 0 or len(kn) == 0:
        return {}
    kn_idx = kn.index.to_numpy()
    ev_idx = ev.index.to_numpy()
    # predict all rows
    all_xy = hw_df[["X", "Y"]].to_numpy(np.float64)
    ancc_hat_all, nn_dist_all = predictor.predict(self_wid, all_xy)

    # calibrate b on the prefix (b = TVT_input + Z - ANCC_hat)
    kn_z = kn["Z"].to_numpy(np.float64)
    kn_tvti = kn["TVT_input"].to_numpy(np.float64)
    ancc_hat_kn = ancc_hat_all[kn_idx - hw_df.index[0]]
    b_vec = kn_tvti + kn_z - ancc_hat_kn
    n_kn = len(kn)
    b_full = float(np.median(b_vec))
    # b_late: last 50 prefix rows
    b_late = float(np.median(b_vec[max(0, n_kn - 50):])) if n_kn >= 5 else b_full
    # drift calibration: extrapolate the MD drift of b from the last 200 rows
    md_kn = kn["MD"].to_numpy(np.float64)
    md_ev = ev["MD"].to_numpy(np.float64)
    md_PS = float(md_kn[-1])
    tail_sl = slice(max(0, n_kn - 200), n_kn)
    b_slope = _robust_slope(md_kn[tail_sl] - md_PS, b_vec[tail_sl])
    md_since = md_ev - md_PS
    b_PS = b_vec[-1] if len(b_vec) > 0 else b_full

    # prefix self-validation
    prefix_resid_rmse = float(np.sqrt(np.mean((kn_tvti - (ancc_hat_kn - kn_z + b_full)) ** 2)))

    # eval row indices (assumes a reset index)
    ri = hw_df.index[0]
    ev_rel = ev_idx - ri
    ev_z = ev["Z"].to_numpy(np.float64)
    ancc_hat_ev = ancc_hat_all[ev_rel]

    tvt_bfull = ancc_hat_ev - ev_z + b_full
    tvt_blate = ancc_hat_ev - ev_z + b_late
    tvt_drift = ancc_hat_ev - ev_z + (b_PS + b_slope * md_since)

    return {
        "kn_idx": kn_idx,
        "ev_idx": ev_idx,
        "ancc_hat_all": ancc_hat_all,
        "nn_dist_all": nn_dist_all,
        "b_full": b_full,
        "b_late": b_late,
        "b_slope": b_slope,
        "b_PS": b_PS,
        "md_PS": md_PS,
        "tvt_bfull": tvt_bfull,
        "tvt_blate": tvt_blate,
        "tvt_drift": tvt_drift,
        "prefix_resid_rmse": prefix_resid_rmse,
        "n_kn": n_kn,
        "n_ev": len(ev),
        "nn_dist_ev": nn_dist_all[ev_rel],
        "md_since": md_since,
    }


# ---------------------------------------------------------------- entry point
def run_spatial_leg(
    test_dir: Path,
    train_dir: Path,
    cfg: dict | None = None,
    guards: bool = True,
) -> tuple[dict, dict]:
    """
    Entry point: produce the spatial TVT_sp for every well in test_dir.

    Kernel-embeddable: no Path(__file__) references. The DenseCloud is always
    rebuilt from train_dir (no cache; takes ~1-2 minutes).

    Args:
      test_dir:  directory of wells to predict
      train_dir: directory the DenseCloud is built from
      cfg:       CONFIG dict (None -> the frozen defaults above)
      guards:    True enables NN_DIST_MAX / PREFIX_RESID_MAX
                 (PREFIX_MIN is always enforced)

    returns:
      id_to_tvt:   {row_id: tvt_blate}
      id_to_valid: {row_id: True/False}  (guard verdict)
    """
    import time as _time
    t0 = _time.time()
    if cfg is None:
        cfg = CONFIG
    b_cal = cfg.get("B_CAL", "late")
    print(f"[spatial_leg] Building DenseCloud from {train_dir} spw={cfg['SPW']}...", flush=True)
    cloud = DenseCloud.build(Path(train_dir), spw=cfg["SPW"])
    print(f"[spatial_leg] Cloud built: {len(cloud.ancc)} pts ({_time.time()-t0:.1f}s)", flush=True)
    pred = make_predictor(cloud, cfg["VARIANT"],
                          k=cfg["K"], power=cfg["POWER"],
                          lam=cfg.get("ANISO_LAMBDA", 1.0),
                          mode=cfg.get("ANISO_MODE", "pca"))
    id_to_tvt: dict[str, float] = {}
    id_to_valid: dict[str, bool] = {}
    n_applied = 0
    for hw_path in sorted(Path(test_dir).glob("*__horizontal_well.csv")):
        wid = hw_path.stem.replace("__horizontal_well", "")
        try:
            hw = pd.read_csv(hw_path)
        except Exception:
            continue
        hw = hw.reset_index(drop=True)
        kn = hw[hw["TVT_input"].notna()]
        ev = hw[hw["TVT_input"].isna()]
        if len(ev) == 0:
            continue
        n_kn = len(kn)
        # PREFIX_MIN is checked regardless of the guards flag
        valid = n_kn >= cfg["PREFIX_MIN"]
        res = spatial_tvt_for_well(hw, pred, wid)
        if not res:
            for i in ev.index:
                id_to_tvt[f"{wid}_{i}"] = float(kn["TVT_input"].iloc[-1]) if n_kn > 0 else 0.0
                id_to_valid[f"{wid}_{i}"] = False
            continue
        # NN_DIST_MAX / PREFIX_RESID_MAX apply only with guards=True
        nn_med = float(np.median(res["nn_dist_ev"])) if len(res["nn_dist_ev"]) else np.inf
        nn_max = float(res["nn_dist_ev"].max()) if len(res["nn_dist_ev"]) else np.inf
        guard_nn = True
        guard_pr = True
        if guards:
            if cfg.get("NN_DIST_MAX") is not None:
                guard_nn = nn_max <= cfg["NN_DIST_MAX"]
                valid = valid and guard_nn
            if cfg.get("PREFIX_RESID_MAX") is not None:
                guard_pr = res["prefix_resid_rmse"] <= cfg["PREFIX_RESID_MAX"]
                valid = valid and guard_pr
        # choose the b calibration
        tvt_sp = res["tvt_blate"] if b_cal == "late" else res["tvt_bfull"]
        # per-well audit log
        guard_rate = float((~np.isfinite(res["nn_dist_ev"]) | (res["nn_dist_ev"] > cfg.get("NN_DIST_MAX", np.inf))).mean()) if guards and cfg.get("NN_DIST_MAX") else 0.0
        print(
            f"[spatial_leg] {wid}: n_kn={n_kn} n_ev={len(ev)} "
            f"b_{b_cal}={res['b_late' if b_cal=='late' else 'b_full']:.2f} "
            f"prefix_resid={res['prefix_resid_rmse']:.2f} "
            f"nn_dist_med={nn_med:.5f} nn_dist_max={nn_max:.5f} "
            f"guard_nn={guard_nn} guard_pr={guard_pr} valid={valid}",
            flush=True,
        )
        for j, i in enumerate(ev.index):
            id_to_tvt[f"{wid}_{i}"] = float(tvt_sp[j])
            id_to_valid[f"{wid}_{i}"] = valid
        if valid:
            n_applied += 1
    print(f"[spatial_leg] Done: {n_applied}/{len(list(Path(test_dir).glob('*__horizontal_well.csv')))} wells valid "
          f"({_time.time()-t0:.1f}s total)", flush=True)
    return id_to_tvt, id_to_valid

## BiGRU residual refiner

Two modules embedded as source strings (executed in their own namespaces so the
feature builder's helpers don't collide with the engine's):

- **Feature builder** — turns each well into a 36-channel sequence tensor on a
  4-ft measured-depth grid: particle-filter posterior statistics (seed paths,
  likelihoods), GR, trajectory, the typewell grid, the beam-search mean, and the
  spatial-surface feature. All normalization is per-well with fixed constants —
  no scaler is ever fit across wells, and the builder never sees the target.
  Same code as `train/gru_features.py`.
- **GRU leg** (`run_gru_leg`) — a PyTorch bidirectional GRU that predicts the
  residual of the particle-filter ensemble against the truth. Inference reuses
  the shared PF cache and averages **15 pretrained models** (5 folds × 3 seeds;
  trained by `train/gru_train.py`). Leg value per row = PF ensemble + residual.

In [ ]:
# ============================================================================
#  BiGRU residual-refiner leg
#  Two modules are embedded as source strings and executed in their own
#  namespaces (the feature builder defines helper names that would otherwise
#  collide with the core engine's).
# ============================================================================
_F102_SRC = r'''
"""GRU feature builder: PF dump dict -> 4ft-grid sequence tensors (36 channels)
for the bidirectional-GRU tail refiner. Leak rules:

  * build_features() DOES NOT take the truth (enforced by a canary at training).
  * All normalization is per-well with fixed constants (25ft / gr_sigma / 1000ft);
    no scaler is ever fit across wells.
  * The only raw-data dependency is the GR NaN gap mask (the dump's gr_ev is
    already interpolated); eval-zone GR observability is legal at inference.

The same module text is used at training time (train/gru_features.py) and
embedded in the submission kernel.
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed


ART = Path(__file__).resolve().parent / "artifacts"
FEAT_DIR = ART / "feat"

CONFIG = dict(
    grid_step=4.0, tvt_scale=25.0, md_scale=1000.0, clip=8.0,
    lik_scale=5.0,                       # pfens_s5 weighting (same as the PF ensemble)
    gr_smooth_ft=25.0, gr_rstd_ft=50.0,
    look_gr_ft=500.0, look_slope_fts=(250.0, 1000.0), dip_trail_ft=200.0,
    grmm_offsets=(-10.0, 0.0, 10.0), frac_band_ft=5.0,
    n_jobs=14, seed=42,
)

# channel registry: (name, group) -- group "cand" = candidate/PF-derived,
# "look" = legal look-ahead, "" = own-well GR/trajectory/position.
CHANNELS = [
    ("pf_anchor_rel", "cand"), ("pf_s3_dis", "cand"), ("pf_s8_dis", "cand"),
    ("pf_s12_dis", "cand"), ("pf_wstd", "cand"), ("pf_q90_10", "cand"),
    ("pf_q75_25", "cand"), ("pf_top1_dis", "cand"), ("pf_top8_dis", "cand"),
    ("pf_frac5", "cand"), ("pf_grmm_m10", "cand"), ("pf_grmm_0", "cand"),
    ("pf_grmm_p10", "cand"), ("pf_ess", "cand"),
    ("gr_norm", ""), ("gr_smooth", ""), ("gr_rstd", ""), ("gr_gap", ""),
    ("z_rel", ""), ("z_slope", ""), ("z_curv", ""),
    ("z_remain", "look"), ("look_slope250", "look"), ("look_slope1000", "look"),
    ("look_gr_mean", "look"), ("look_gr_std", "look"),
    ("node_dip", "cand"), ("ir", ""),
    ("md_since", ""), ("dist_end", ""), ("idx_norm", ""), ("total_len", ""),
    ("fam_sp_blate", "cand"), ("fam_beam_mean", "cand"),
    ("fam_poly1_t500", "cand"), ("fam_flat", "cand"),
]
CH_IDX = {n: i for i, (n, _) in enumerate(CHANNELS)}
CAND_CH = [i for i, (_, g) in enumerate(CHANNELS) if g == "cand"]
LOOK_CH = [i for i, (_, g) in enumerate(CHANNELS) if g == "look"]
ANCHOR_CH = CH_IDX["pf_anchor_rel"]        # crop re-anchoring rebases this channel
N_CH = len(CHANNELS)


def _fwd_mean(v, k):
    """out[i] = mean(v[i:i+k]) with shrinking window at the tail."""
    n = len(v)
    c = np.concatenate([[0.0], np.cumsum(v, dtype=np.float64)])
    hi = np.minimum(np.arange(n) + k, n)
    cnt = hi - np.arange(n)
    return (c[hi] - c[:-1]) / np.maximum(cnt, 1)


def _fwd_std(v, k):
    m1 = _fwd_mean(v, k)
    m2 = _fwd_mean(v * v, k)
    return np.sqrt(np.maximum(m2 - m1 * m1, 0.0))


def _smooth(v, k):
    """centered boxcar mean, width k samples (edge-padded)."""
    if k <= 1:
        return v.astype(np.float64)
    pad = k // 2
    vp = np.pad(v.astype(np.float64), pad, mode="edge")
    c = np.concatenate([[0.0], np.cumsum(vp)])
    out = (c[k:] - c[:-k]) / k
    return out[: len(v)]


def build_features(d, gr_raw_nan):
    """d: dict of dump arrays WITHOUT tvt_true. gr_raw_nan: raw eval-zone GR (NaN=gap).

    Returns X (C,L) float32, md_g (L,), base_g (L,) float64.
    """
    cfg = CONFIG
    names = [str(n) for n in d["names"]]
    ni = {n: i for i, n in enumerate(names)}
    cands = d["cands"].astype(np.float64)
    md = d["md_ev"].astype(np.float64)
    z = d["z_ev"].astype(np.float64)
    gr = d["gr_ev"].astype(np.float64)
    liks = d["liks"].astype(np.float64)
    gg = d["gg"].astype(np.float64)
    gmin = float(d["gmin"]); gstep = float(d["gstep"])
    last_tvt = float(d["last_tvt"]); last_md = float(d["last_md"])
    ir = float(d["ir"]); gr_sigma = float(d["gr_sigma"])

    base = cands[ni["pfens_s5"]]
    seeds = cands[[ni[f"pf{s:03d}"] for s in range(len(liks))]]     # (S, n_ev)
    w = np.exp((liks - liks.max()) / cfg["lik_scale"]); w /= w.sum()

    # ---- 4ft grid ----
    step = cfg["grid_step"]
    md_g = np.arange(md[0], md[-1], step)
    if len(md_g) == 0 or md[-1] - md_g[-1] > 0.5:
        md_g = np.append(md_g, md[-1])
    L = len(md_g)

    def lin(v):
        return np.interp(md_g, md, v)

    med_dmd = float(np.median(np.diff(md))) if len(md) > 1 else 1.0
    k_native = max(1, int(round(step / max(med_dmd, 1e-6))))

    base_g = lin(base)
    z_g = lin(z)
    gr_g = lin(_smooth(gr, k_native))                # anti-aliased GR on grid
    gr_med = float(np.median(gr))

    ts, ms, cl = cfg["tvt_scale"], cfg["md_scale"], cfg["clip"]
    X = np.zeros((N_CH, L), dtype=np.float64)

    def put(name, v):
        X[CH_IDX[name]] = v

    # ---- PF posterior block ----
    put("pf_anchor_rel", (base_g - last_tvt) / ts)
    for sc, nm in ((3, "pf_s3_dis"), (8, "pf_s8_dis"), (12, "pf_s12_dis")):
        put(nm, (lin(cands[ni[f"pfens_s{sc}"]]) - base_g) / ts)
    wstd = np.sqrt(np.maximum((w[:, None] * (seeds - base[None, :]) ** 2).sum(0), 0.0))
    put("pf_wstd", lin(wstd) / ts)
    q10, q25, q75, q90 = np.percentile(seeds, [10, 25, 75, 90], axis=0)
    put("pf_q90_10", lin(q90 - q10) / ts)
    put("pf_q75_25", lin(q75 - q25) / ts)
    put("pf_top1_dis", (lin(cands[ni["pf_top1lik"]]) - base_g) / ts)
    put("pf_top8_dis", (lin(cands[ni["pf_top8lik"]]) - base_g) / ts)
    put("pf_frac5", lin((np.abs(seeds - base[None, :]) <= cfg["frac_band_ft"]).mean(0)))
    tw_axis = gmin + gstep * np.arange(len(gg))
    for dlt, nm in zip(cfg["grmm_offsets"], ("pf_grmm_m10", "pf_grmm_0", "pf_grmm_p10")):
        mm = (gr - np.interp(base + dlt, tw_axis, gg)) / gr_sigma
        put(nm, lin(_smooth(mm, k_native)))
    put("pf_ess", np.full(L, 1.0 / (np.sum(w ** 2) * len(w))))

    # ---- GR block ----
    grn_native = (gr - gr_med) / gr_sigma
    put("gr_norm", (gr_g - gr_med) / gr_sigma)
    put("gr_smooth", lin(_smooth(grn_native, max(1, int(round(cfg["gr_smooth_ft"] / max(med_dmd, 1e-6)))))))
    k_rstd = max(2, int(round(cfg["gr_rstd_ft"] / max(med_dmd, 1e-6))))
    pad = k_rstd // 2
    gp = np.pad(grn_native, pad, mode="edge")
    c1 = np.concatenate([[0.0], np.cumsum(gp)])
    c2 = np.concatenate([[0.0], np.cumsum(gp * gp)])
    m1 = (c1[k_rstd:] - c1[:-k_rstd]) / k_rstd
    m2 = (c2[k_rstd:] - c2[:-k_rstd]) / k_rstd
    put("gr_rstd", lin(np.sqrt(np.maximum(m2 - m1 * m1, 0.0))[: len(gr)]))
    gap = np.isnan(gr_raw_nan).astype(np.float64)
    put("gr_gap", lin(_smooth(gap, k_native)))

    # ---- trajectory + legal look-ahead ----
    put("z_rel", (z_g - z_g[0]) / 100.0)
    slope_z = np.gradient(_smooth(z_g, 5), md_g)
    put("z_slope", slope_z * 20.0)
    put("z_curv", np.gradient(slope_z, md_g) * 2000.0)
    put("z_remain", (z_g[-1] - z_g) / 100.0)
    for W, nm in zip(cfg["look_slope_fts"], ("look_slope250", "look_slope1000")):
        k = max(2, int(round(W / step)))
        dz_f = _fwd_mean(np.gradient(z_g, md_g), k)
        put(nm, dz_f * 20.0)
    k_gr = max(2, int(round(cfg["look_gr_ft"] / step)))
    grn_g = (gr_g - gr_med) / gr_sigma
    put("look_gr_mean", _fwd_mean(grn_g, k_gr))
    put("look_gr_std", _fwd_std(grn_g, k_gr))
    k_dip = max(1, int(round(cfg["dip_trail_ft"] / step)))
    dip = np.empty(L)
    for i in range(L):
        j = max(0, i - k_dip)
        dm = md_g[i] - md_g[j]
        dip[i] = (base_g[i] - base_g[j]) / dm if dm > 0 else 0.0
    put("node_dip", dip * 20.0)
    put("ir", np.full(L, ir * 50.0))

    # ---- positional ----
    put("md_since", (md_g - last_md) / ms)
    put("dist_end", (md_g[-1] - md_g) / ms)
    put("idx_norm", np.arange(L) / max(L - 1, 1))
    put("total_len", np.full(L, (md_g[-1] - md_g[0]) / ms))

    # ---- alternative-family disagreement ----
    for src, nm in (("sp_blate", "fam_sp_blate"), ("beam_mean", "fam_beam_mean"),
                    ("poly1_t500", "fam_poly1_t500"), ("flat_hold", "fam_flat")):
        put(nm, (lin(cands[ni[src]]) - base_g) / ts)

    np.clip(X, -cl, cl, out=X)
    return X.astype(np.float32), md_g, base_g


'''
_f102_ns = {'__name__': 'gru_features', '__file__': 'gru_features.py'}
exec(compile(_F102_SRC, 'gru_features.py', 'exec'), _f102_ns)

_GRU_SRC = r'''
"""GRU refinement leg (kernel-portable).

Runs a 128-seed PF pass over the test wells (cache-shared with the core
engine's pass, so the filter effectively runs once), reconstructs the per-well
dump dict the feature builder expects, builds the 36-channel features, and
predicts residuals with the 15-model BiGRU ensemble.
Leg value per row = pfens_s5 + mean residual.

Dependency injection (works both in-kernel and locally):
  deps = dict(p128_run_pf=..., p128_gr_sigma=..., build_features=...,
              beam_family=..., zero_ch=[...], n_ch=36)
In-kernel these come from the core-engine cell and the embedded feature builder.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

GCONFIG = dict(n_seeds=128, lik_scales=(3.0, 5.0, 8.0, 12.0), gg_step=0.2,
               n_models=15)


# ---- candidate-library ports (the subset the 36ch builder reads) ------------
def _typewell_grid(tw_tvt, tw_gr, step=0.2):
    """Typewell GR resampled on a regular TVT grid."""
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax + step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)


def _pf_derived(paths, liks, scales):
    """Likelihood-weighted PF ensembles (pfens_s*) + top1/top8 paths."""
    out = {}
    for sc in scales:
        w = np.exp((liks - liks.max()) / sc); w /= w.sum()
        out[f"pfens_s{sc:g}"] = (w[:, None] * paths).sum(0).astype(np.float32)
    order = np.argsort(liks)[::-1]
    out["pf_top1lik"] = paths[order[0]].copy()
    out["pf_top8lik"] = paths[order[:8]].mean(0).astype(np.float32)
    return out


def _trivial_subset(hw):
    """Trivial prefix extrapolations, reduced to the channels the builder reads."""
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    ev_md = ev["MD"].values.astype(float)
    last_tvt = float(kn["TVT_input"].iloc[-1])
    out = {"flat_hold": np.full(len(ev), last_tvt, np.float32)}
    t = kn.tail(500)
    x = t["MD"].values.astype(float); y = t["TVT_input"].values.astype(float)
    if len(t) < 10 or np.std(x) < 1e-6:
        out["poly1_t500"] = np.full(len(ev), last_tvt, np.float32)
    else:
        c = np.polyfit(x, y, 1)
        out["poly1_t500"] = np.polyval(c, ev_md).astype(np.float32)
    return out


def _prefix_ir(hw):
    """Prefix dip rate of (TVT+Z), same estimator as the PF init."""
    kn = hw[hw["TVT_input"].notna()]
    tail = kn.tail(30)
    dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values)
    dm = np.diff(tail["MD"].values); m = dm > 0
    return float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0


def build_dump_equiv(hw, tw_tvt, tw_gr, seed_paths_full, liks, gr_sigma, sp_ev=None,
                     beam_ev=None):
    """Reconstruct the dump dict fields the feature builder reads.
    seed_paths_full: (S, len(hw)) full-column p128 paths.
    sp_ev: optional spatial path (n_ev,) for the sp_blate channel (in the kernel
    it comes from run_spatial_leg guards=off); non-finite/missing rows fall back
    to pfens_s5 (channel value 0). None -> zeros.
    Returns (d, gr_raw_ev)."""
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    ev_idx = ev.index.values
    P = seed_paths_full[:, ev_idx].astype(np.float32)
    cands = {f"pf{s:03d}": P[s] for s in range(P.shape[0])}
    cands.update(_pf_derived(P, liks, GCONFIG["lik_scales"]))
    n_ev = len(ev_idx)
    if sp_ev is None:
        cands["sp_blate"] = np.zeros(n_ev, np.float32)
    else:
        pf5 = cands["pfens_s5"].astype(np.float64)
        sp = np.asarray(sp_ev, np.float64)
        cands["sp_blate"] = np.where(np.isfinite(sp), sp, pf5).astype(np.float32)
    if beam_ev is None:
        cands["beam_mean"] = np.zeros(n_ev, np.float32)   # zeroed variant
    else:
        cands["beam_mean"] = np.asarray(beam_ev, np.float32)
    cands.update(_trivial_subset(hw))
    names = list(cands)
    gg, gmin, gstep = _typewell_grid(tw_tvt, tw_gr, GCONFIG["gg_step"])
    gr_i = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean())
    d = dict(
        names=np.array(names),
        cands=np.stack([cands[k] for k in names], 0).astype(np.float32),
        md_ev=ev["MD"].values.astype(np.float32),
        z_ev=ev["Z"].values.astype(np.float32),
        gr_ev=gr_i.values.astype(np.float32)[ev_idx],
        gg=gg.astype(np.float32), gmin=np.float64(gmin), gstep=np.float64(gstep),
        liks=liks.astype(np.float64),
        last_tvt=np.float64(float(kn["TVT_input"].iloc[-1])),
        last_md=np.float64(float(kn["MD"].iloc[-1])),
        ir=np.float64(_prefix_ir(hw)),
        gr_sigma=np.float64(gr_sigma),
    )
    gr_raw_ev = hw["GR"].to_numpy(float)[ev_idx]
    return d, gr_raw_ev


def gru_features_one(hw, tw_tvt, tw_gr, deps, sp_ev=None):
    """PF 128 seeds -> features. Returns (X, md_g, ev_index, base_nat, nat_md)."""
    n = GCONFIG["n_seeds"]
    res = [deps["p128_run_pf"](hw, tw_tvt, tw_gr, seed=s) for s in range(n)]
    paths = np.stack([r[0] for r in res]); liks = np.array([r[1] for r in res])
    gr_sigma = deps["p128_gr_sigma"](hw, tw_tvt, tw_gr)
    beam_ev = None
    if deps.get("beam_family") is not None:
        beam_ev = deps["beam_family"](hw, tw_tvt, tw_gr)["beam_mean"]
    d, gr_raw_ev = build_dump_equiv(hw, tw_tvt, tw_gr, paths, liks, gr_sigma, sp_ev,
                                    beam_ev)
    X, md_g, _base_g = deps["build_features"](d, gr_raw_ev)
    X[deps["zero_ch"]] = 0.0
    ni = [str(x) for x in d["names"]].index("pfens_s5")
    base_nat = d["cands"][ni].astype(np.float64)
    ev = hw[hw["TVT_input"].isna()]
    return X, md_g, ev.index.values, base_nat, d["md_ev"].astype(np.float64)


def _torch_models(weight_files, n_ch):
    import torch
    import torch.nn as nn
    from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

    class RefinerGRU(nn.Module):          # same architecture as train/gru_model.py
        def __init__(self, n_ch, hidden=64, drop=0.2):
            super().__init__()
            self.frontend = nn.Sequential(nn.Conv1d(n_ch, 64, 5, padding=2), nn.GELU())
            self.gru = nn.GRU(64, hidden, num_layers=2, batch_first=True,
                              bidirectional=True, dropout=drop)
            self.head = nn.Sequential(nn.Linear(2 * hidden, 64), nn.GELU(),
                                      nn.Dropout(drop), nn.Linear(64, 1))

        def forward(self, x, lengths):
            h = self.frontend(x).transpose(1, 2)
            packed = pack_padded_sequence(h, lengths.cpu(), batch_first=True,
                                          enforce_sorted=False)
            out, _ = self.gru(packed)
            out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.shape[-1])
            return self.head(out).squeeze(-1) * 25.0

    models = []
    for wf in weight_files:
        m = RefinerGRU(n_ch)
        m.load_state_dict(torch.load(wf, map_location="cpu"))
        m.eval()
        models.append(m)
    return models


def gru_residual(models, X):
    """Mean residual over the model ensemble for one well. X: (C, L) float32."""
    import torch
    with torch.no_grad():
        xb = torch.from_numpy(X[None].astype(np.float32))
        L = torch.tensor([X.shape[1]])
        rs = [m(xb, L)[0].numpy().astype(np.float64) for m in models]
    return np.mean(rs, axis=0)


def run_gru_leg(test_paths, test_dir, weight_files, deps, n_jobs=4, verbose=True,
                sp_map=None):
    """Full leg over test wells -> {row_id: gru_leg_value}. Never raises per-well
    (a failed well is skipped -> falls back to base blend for those rows).
    sp_map: optional {row_id: spatial_value} from run_spatial_leg(guards=off)."""
    from joblib import Parallel, delayed

    def _one(hp):
        try:
            wid = hp.stem.replace("__horizontal_well", "")
            tp = test_dir / f"{wid}__typewell.csv"
            if not tp.exists():
                return None
            hw = pd.read_csv(hp)
            tw = pd.read_csv(tp).sort_values("TVT")
            ev = hw[hw["TVT_input"].isna()]
            kn = hw[hw["TVT_input"].notna()]
            if len(ev) == 0 or len(kn) == 0:
                return None
            tw_tvt = tw["TVT"].to_numpy(float)
            tw_gr = tw["GR"].fillna(tw["GR"].mean()).to_numpy(float)
            if len(tw.dropna(subset=["TVT"])) < 10:
                return None
            sp_ev = None
            if sp_map is not None:
                sp_ev = np.array([sp_map.get(f"{wid}_{i}", np.nan) for i in ev.index],
                                 np.float64)
            X, md_g, ev_idx, base_nat, nat_md = gru_features_one(
                hw, tw_tvt, tw_gr, deps, sp_ev)
            return wid, X, md_g, ev_idx, base_nat, nat_md
        except Exception as e:  # noqa: BLE001 -- per-well fallback, never kill the sub
            print(f"[gru_leg] well {hp.stem} FAILED ({type(e).__name__}: {e})", flush=True)
            return None

    feats = Parallel(n_jobs=n_jobs, prefer="processes")(delayed(_one)(hp) for hp in test_paths)
    feats = [f for f in feats if f is not None]
    if verbose:
        print(f"[gru_leg] features built for {len(feats)}/{len(test_paths)} wells", flush=True)
    models = _torch_models(weight_files, deps["n_ch"])
    out = {}
    for wid, X, md_g, ev_idx, base_nat, nat_md in feats:
        r_g = gru_residual(models, X)
        r_nat = np.interp(nat_md, md_g.astype(np.float64), r_g)
        vals = base_nat + r_nat
        for i, v in zip(ev_idx, vals):
            out[f"{wid}_{i}"] = float(v)
    if verbose:
        print(f"[gru_leg] {len(out)} rows predicted ({len(models)} models)", flush=True)
    return out
'''
_gru_ns = {'__name__': 'gru_leg'}
exec(compile(_GRU_SRC, 'gru_leg.py', 'exec'), _gru_ns)
run_gru_leg = _gru_ns['run_gru_leg']


def beam_family(hw, tw_tvt, tw_gr):
    """Mean of the 7-config beam-search paths (the fam_beam_mean channel).
    Reuses the core engine's beam_search / BEAMS."""
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    last_tvt = float(kn["TVT_input"].iloc[-1])
    gr_ev = hw["GR"].interpolate(limit_direction="both").values.astype(float)[ev.index]
    paths = [beam_search(gr_ev, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
             for bs, mc, es, r, _tag in BEAMS]
    return {"beam_mean": np.stack(paths, 0).mean(0).astype(np.float32)}


GRU_ZERO_CH = []
GRU_DEPS = dict(p128_run_pf=p128_run_pf, p128_gr_sigma=p128_gr_sigma,
                build_features=_f102_ns['build_features'],
                zero_ch=GRU_ZERO_CH, n_ch=_f102_ns['N_CH'],
                beam_family=beam_family)


def _find_gru_weights():
    import glob as _gg
    _ws = sorted(_gg.glob('/kaggle/input/**/*_f*_s*.pt', recursive=True))
    if not _ws:  # local run: weights produced by train/gru_train.py
        _ws = sorted(_gg.glob(str(_HERE / 'train' / 'artifacts' / 'gru' / 'weights' / '*_f*_s*.pt')))
    assert len(_ws) == 15, 'gru weights found %d (want 15)' % len(_ws)
    return _ws


print('[gru] leg loaded n_ch=%d' % _f102_ns['N_CH'])

## Duplicate-well recovery

Defines `apply_dup_hedge`. The hidden test contains byte-identical copies of
training wells — including renamed ones. Candidates are generated three ways
(same ID, lateral-XY proximity, GR signature) and accepted only when they pass
three strict gates: prefix TVT RMSE < 0.02 ft, full-length GR agreement, and
full-length Z agreement (this rejects sidetrack twins). Because the organizers
re-issued labels, a pure copy of the training truth hurts; the applied value is a
50/50 hedge between the transferred truth and the model prediction.

In [ ]:
"""Duplicate-well recovery (exact-match recovery + interpretation hedge).

The hidden test contains byte-identical copies of training wells (proven via
the leaderboard). Training files carry the ground-truth TVT, so on a well whose
duplication is *proven*, transferring the training TVT curve is the best
possible prediction.

The organizers however re-issued labels, so a pure copy (w=1.0) hurts. Hence
the **hedge**:
    tvt_final = XR_W_XFER * (transferred train TVT) + (1 - XR_W_XFER) * model_pred
mixing the two independent interpretations; w = 0.5 is the safe production value.

Candidate generation (3 routes; also finds renamed/shifted duplicates):
  (a) same well id
  (b) lateral mean-XY proximity
  (c) GR signature match (row count + rounded GR sum)
Gates (a candidate must pass ALL of them -- strong proof that rejects
sidetrack twins):
  gate1: known-prefix TVT_input vs train TVT -> RMSE < 0.02 ft
  gate2: full-length GR vs train GR -> mean|d| < 0.50 API (twin guard)
  gate3: full-length Z vs train Z -> mean|d| < 0.02 ft
  min_visible = 50 (overlapping rows)

Acceptance prefers the same-id candidate: if it passes the gates it is always
taken; content candidates are used only when no same-id candidate exists or it
fails (renamed test wells whose labels were re-interpreted are a trap for pure
content matching).
"""
from __future__ import annotations
import os as _os, glob as _glob, time as _time
import numpy as np
import pandas as pd
from pathlib import Path

# ===== CONFIG ===============================================================
XR_GATES = dict(tvt_rmse=0.02, gr_mad=0.50, z_mad=0.02, min_visible=50)
XR_W_XFER = 0.5      # transfer weight (1.0=full override, 0.5=hedge, 0.0=audit only)
XR_XY_KNN = 8        # candidates from lateral mean-XY proximity
XR_GR_SIG_DECIMALS = 0  # rounding of the GR sum for the signature match
# ==========================================================================


def _interp(md_to, md_from, vals):
    """Linearly transfer a train curve onto the test MD grid (NaN outside)."""
    return np.interp(md_to, md_from, vals, left=np.nan, right=np.nan)


def _gate_report(te: pd.DataFrame, tr: pd.DataFrame) -> tuple[bool, dict]:
    """Run the 3 gates; returns (ok, diagnostics dict)."""
    md = te["MD"].to_numpy(float)
    rep: dict = {}
    # gate 3: full-length trajectory Z (covers the hidden lateral too)
    z_tr = _interp(md, tr["MD"].to_numpy(float), tr["Z"].to_numpy(float))
    m = np.isfinite(z_tr) & te["Z"].notna().to_numpy()
    if m.sum() < XR_GATES["min_visible"]:
        return False, {"reason": "no overlap"}
    rep["z_mad"] = float(np.mean(np.abs(te["Z"].to_numpy(float)[m] - z_tr[m])))
    rep["cover"] = float(m.mean())
    # gate 2: full-length GR (eval-zone GR is visible -> sidetrack guard)
    gr_tr = _interp(md, tr["MD"].to_numpy(float),
                    tr["GR"].interpolate(limit_direction="both").to_numpy(float))
    g = m & te["GR"].notna().to_numpy() & np.isfinite(gr_tr)
    rep["gr_mad"] = (float(np.mean(np.abs(te["GR"].to_numpy(float)[g] - gr_tr[g])))
                     if g.any() else np.inf)
    # gate 1: the known-prefix TVT must reproduce the train TVT almost exactly
    tvt_tr = _interp(md, tr["MD"].to_numpy(float), tr["TVT"].to_numpy(float))
    v = te["TVT_input"].notna().to_numpy() & np.isfinite(tvt_tr)
    if v.sum() < XR_GATES["min_visible"]:
        return False, {"reason": "no visible TVT", **rep}
    d = te["TVT_input"].to_numpy(float)[v] - tvt_tr[v]
    rep["tvt_rmse"] = float(np.sqrt(np.mean(d * d)))
    rep["n_visible"] = int(v.sum())
    ok = (rep["tvt_rmse"] < XR_GATES["tvt_rmse"]
          and rep["gr_mad"] < XR_GATES["gr_mad"]
          and rep["z_mad"] < XR_GATES["z_mad"])
    return ok, rep


def _build_index(train: dict[str, pd.DataFrame]):
    """Build the signature index over the training wells for candidate generation."""
    xy = []   # (wid, meanX, meanY)
    gr_sig: dict = {}  # (n_rows, round(sum GR)) -> [wid,...]
    for wid, df in train.items():
        xy.append((wid, float(df["X"].mean()), float(df["Y"].mean())))
        key = (len(df), round(float(np.nansum(df["GR"].to_numpy())), XR_GR_SIG_DECIMALS))
        gr_sig.setdefault(key, []).append(wid)
    xy_arr = np.array([(x, y) for _, x, y in xy], dtype=float)
    xy_ids = [w for w, _, _ in xy]
    return xy_ids, xy_arr, gr_sig


def _candidates(wid: str, te: pd.DataFrame, train: dict,
                xy_ids, xy_arr, gr_sig, exclude: set | None = None) -> list[str]:
    """Generate candidate well ids by the 3 routes (same id first, deduplicated)."""
    exclude = exclude or set()
    cand: list[str] = []
    seen: set = set()

    def _add(w):
        if w and w not in seen and w not in exclude and w in train:
            seen.add(w); cand.append(w)

    # (a) same id
    if wid in train:
        _add(wid)
    # (b) lateral mean-XY proximity
    try:
        tx, ty = float(te["X"].mean()), float(te["Y"].mean())
        if np.isfinite(tx) and np.isfinite(ty) and len(xy_arr):
            d2 = (xy_arr[:, 0] - tx) ** 2 + (xy_arr[:, 1] - ty) ** 2
            for j in np.argsort(d2)[:XR_XY_KNN]:
                _add(xy_ids[j])
    except Exception:
        pass
    # (c) GR signature (row count + GR sum)
    try:
        key = (len(te), round(float(np.nansum(te["GR"].to_numpy())), XR_GR_SIG_DECIMALS))
        for w in gr_sig.get(key, []):
            _add(w)
    except Exception:
        pass
    return cand


def apply_dup_hedge(submission_df: pd.DataFrame, test_dir, train_dir,
                    w_xfer: float = XR_W_XFER, log=print) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply the duplicate-well hedge to a submission.

    submission_df: columns ["id","tvt"]; id is "{well}_{row_idx}".
    test_dir / train_dir: the test/train well CSV directories.
    Returns (new submission_df, audit_df); the input is not mutated.
    """
    t0 = _time.time()
    test_dir = Path(test_dir); train_dir = Path(train_dir)
    sub = submission_df.copy()
    sub["well"] = sub["id"].str.rsplit("_", n=1).str[0]
    sub["row_idx"] = sub["id"].str.rsplit("_", n=1).str[1].astype(int)

    # index the training wells in one pass (needed columns only, float64)
    cols = ["MD", "X", "Y", "Z", "GR", "TVT"]
    train: dict[str, pd.DataFrame] = {}
    tfiles = sorted(_glob.glob(str(train_dir / "*__horizontal_well.csv")))
    for f in tfiles:
        wid = _os.path.basename(f).split("__")[0]
        df = pd.read_csv(f, usecols=lambda c: c in cols)
        train[wid] = df.astype(np.float64)
    xy_ids, xy_arr, gr_sig = _build_index(train)
    log(f"[dup_hedge] indexed {len(train)} train wells in {_time.time()-t0:.0f}s")

    audit: list[dict] = []
    pred = dict(zip(sub["id"], sub["tvt"].astype(float)))
    for wid, gdf in sub.groupby("well", sort=True):
        row = {"well": wid, "accepted": "", "rows_overridden": 0}
        try:
            tpath = test_dir / f"{wid}__horizontal_well.csv"
            if not tpath.exists():
                row["reason"] = "no test file"; audit.append(row); continue
            te = pd.read_csv(tpath)
            cand = _candidates(wid, te, train, xy_ids, xy_arr, gr_sig)
            row["n_candidates"] = len(cand)
            # same-id candidate first; ties broken by smallest tvt_rmse.
            best, best_rep = None, None
            for c in sorted(cand, key=lambda c: c != wid):
                ok, rep = _gate_report(te, train[c])
                if ok:
                    if best is None or rep["tvt_rmse"] < best_rep["tvt_rmse"]:
                        best, best_rep = c, rep
                    if c == wid:
                        break  # same id passed -> final (the safe scope)
            if best is None:
                row["reason"] = "no candidate passed gates"
                audit.append(row); continue
            tr = train[best]
            tvt = _interp(te["MD"].to_numpy(float), tr["MD"].to_numpy(float),
                          tr["TVT"].to_numpy(float))
            n_row = 0
            for rid, ri in zip(gdf["id"], gdf["row_idx"]):
                if 0 <= ri < len(tvt) and np.isfinite(tvt[ri]):
                    pred[rid] = w_xfer * float(tvt[ri]) + (1.0 - w_xfer) * pred[rid]
                    n_row += 1
            row.update(accepted=best, rows_overridden=n_row, **best_rep)
        except Exception as e:                       # defensive: never break the submission
            row["reason"] = f"error: {e}"
        audit.append(row)

    if w_xfer > 0.0:
        sub["tvt"] = sub["id"].map(pred).astype(float)
    else:
        log("[dup_hedge] DIAGNOSTIC mode (w_xfer=0): submission untouched")
    audit_df = pd.DataFrame(audit)
    n_ok = int((audit_df.get("accepted", pd.Series([], dtype=object)).astype(str) != "").sum()) \
        if len(audit_df) else 0
    n_over = int(audit_df["rows_overridden"].sum()) if len(audit_df) else 0
    log(f"[dup_hedge] {n_ok}/{sub['well'].nunique()} wells recovered, "
        f"{n_over} rows overridden, w_xfer={w_xfer}, {_time.time()-t0:.0f}s total")
    return sub[["id", "tvt"]], audit_df

## Contact override

Defines `apply_contact_override`: rebuilds TVT directly from a formation-contact
elevation (`tvt = ref_tvt − (Z − contact)`, with an offset correction) and
replaces predictions only on wells where the reconstruction reproduces the known
prefix within 1 ft RMSE — a self-validation gate that keeps it silent whenever
the geometry does not check out.

In [ ]:
"""Guarded contact override.

Rebuilds TVT directly from a formation-contact elevation and applies it to the
submission as a post-process.

Mechanism:
  Take the TVT reference of the EGFDU contact from the typewell (ref_tvt) and
  compute, for every row of the horizontal well,
      tvt_raw = ref_tvt - (Z - EGFDU_contact)
  then offset-correct to get a contact-derived TVT curve.
  Fire only on wells whose visible prefix self-validates within a threshold,
  and fully replace the predicted rows inside the covered MD range.

Why guard C is safe against re-issued labels:
  the duplicate-well hedge requires TVT_input to reproduce the train TVT almost
  exactly (RMSE < 0.02 ft). The contact override intentionally runs with a
  looser guard (CO_KNOWN_RMSE = 1.0 ft), but on "re-issued" wells whose
  TVT_input is subtly shifted, guard C fails -> zero wells fire -> no-op.
"""

from __future__ import annotations
import glob as _glob, os as _os, time as _time
import numpy as np
import pandas as pd
from pathlib import Path

# ===== CONFIG ================================================================
CO_MIN_CONTACT = 100   # guard A: minimum rows with a valid contact
CO_MIN_KNOWN   = 50    # guard B: minimum visible-prefix rows inside the train MD range
CO_KNOWN_RMSE  = 1.0   # guard C: known-prefix self-validation RMSE cap (ft)
# =============================================================================

def _rmse(a, b) -> float:
    """RMSE over pairs where both values are finite; inf when no valid pair."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() == 0:
        return float("inf")
    d = a[m] - b[m]
    return float(np.sqrt(np.mean(d * d)))


def tvt_from_contacts(hw_tr: pd.DataFrame, tw_tr: pd.DataFrame) -> np.ndarray:
    """Rebuild the TVT curve from contact elevations.

    hw_tr: train horizontal-well CSV (raw pd.read_csv; not sorted)
    tw_tr: train typewell CSV (raw pd.read_csv; not sorted)
    returns: a TVT array with the same row count as hw_tr (float, may hold NaN)
    """
    tw_g = tw_tr.dropna(subset=["Geology"])
    ref_col = "EGFDU"
    ref_tvt = tw_g.loc[tw_g["Geology"] == "EGFDU", "TVT"].min()
    if pd.isna(ref_tvt):
        # no EGFDU -> use the first Geology in file order (do not sort)
        ref_col = tw_g["Geology"].iloc[0]
        ref_tvt = tw_g.loc[tw_g["Geology"] == ref_col, "TVT"].min()
    # a missing ref_col column raises KeyError -> the caller skips the well
    tvt_raw = ref_tvt - (hw_tr["Z"] - hw_tr[ref_col])
    offset = (hw_tr["TVT"] - tvt_raw).mean()  # pandas mean skips NaN
    return (tvt_raw + offset).to_numpy(dtype=float)


def apply_contact_override(
    sub: pd.DataFrame,
    test_dir,
    train_dir,
    log=print,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply the contact override to a submission.

    sub: DataFrame with columns ["id","tvt"]; id is "{well}_{row_idx}".
         Row order and the id column are never changed (a copy is returned).
    test_dir / train_dir: the test/train well CSV directories.
    Returns (new_sub, audit_df); the input is not mutated.
    """
    test_dir = Path(test_dir)
    train_dir = Path(train_dir)

    # id -> tvt lookup for the submission
    pred = dict(zip(sub["id"], sub["tvt"].astype(float)))

    # pre-group the submission by well (O(n); O(1) lookups in the loop)
    sub_ext = sub.copy()
    sub_ext["_well"] = sub_ext["id"].str.rsplit("_", n=1).str[0]
    sub_ext["_ri"] = sub_ext["id"].str.rsplit("_", n=1).str[1].astype(int)
    sub_by_well: dict[str, list[tuple[str, int]]] = {
        wid: list(zip(g["id"], g["_ri"]))
        for wid, g in sub_ext.groupby("_well", sort=False)
    }

    audit: list[dict] = []
    n_fired_total = 0
    n_rows_total = 0

    for te_path in sorted(_glob.glob(str(test_dir / "*__horizontal_well.csv"))):
        wid = _os.path.basename(te_path).split("__")[0]
        row: dict = {
            "well": wid, "accepted": "", "paired_train": "", "rows_overridden": 0,
            "reason": "", "n_contact": 0, "n_known": 0,
            "known_rmse": float("nan"), "mean_abs_delta": float("nan"),
            "max_abs_delta": float("nan"),
        }

        # the train reference is the same-id well
        tr_hw_path = train_dir / f"{wid}__horizontal_well.csv"
        tr_tw_path = train_dir / f"{wid}__typewell.csv"

        if tr_hw_path.exists() and tr_tw_path.exists():
            train_wid = wid
            row["paired_train"] = wid
        else:
            row["reason"] = "no_train_file"
            audit.append(row)
            continue

        try:
            hw_tr = pd.read_csv(tr_hw_path)
            tw_tr = pd.read_csv(tr_tw_path)

            phys = tvt_from_contacts(hw_tr, tw_tr)
            md_raw = hw_tr["MD"].to_numpy(float)

            # guard A: rows with a valid contact
            m = np.isfinite(phys) & np.isfinite(md_raw)
            row["n_contact"] = int(m.sum())
            if row["n_contact"] < CO_MIN_CONTACT:
                row["reason"] = "few_contact"
                audit.append(row)
                continue

            order = np.argsort(md_raw[m])
            md_tr = md_raw[m][order]
            ph_tr = phys[m][order]

            # test horizontal well (row order untouched)
            hw_te = pd.read_csv(test_dir / f"{wid}__horizontal_well.csv")

            # guard B: visible-prefix rows inside the train MD range
            kn_mask = hw_te["TVT_input"].notna()
            in_range = (hw_te["MD"] >= md_tr[0]) & (hw_te["MD"] <= md_tr[-1])
            kn = hw_te[kn_mask & in_range]
            row["n_known"] = len(kn)
            if row["n_known"] < CO_MIN_KNOWN:
                row["reason"] = "few_known"
                audit.append(row)
                continue

            # guard C: known-prefix RMSE (contact-derived vs TVT_input)
            rk = _rmse(
                np.interp(kn["MD"].to_numpy(float), md_tr, ph_tr),
                kn["TVT_input"].to_numpy(float),
            )
            row["known_rmse"] = rk
            if not np.isfinite(rk) or rk > CO_KNOWN_RMSE:
                row["reason"] = "prefix_mismatch"
                audit.append(row)
                continue

            # fire: replace this well's submission rows inside the MD range
            md_te = hw_te["MD"].to_numpy(float)
            well_rows = sub_by_well.get(wid, [])

            deltas: list[float] = []
            n_over = 0
            for rid, ri in well_rows:
                if 0 <= ri < len(hw_te) and md_tr[0] <= md_te[ri] <= md_tr[-1]:
                    new_val = float(np.interp(md_te[ri], md_tr, ph_tr))
                    deltas.append(abs(new_val - pred[rid]))
                    pred[rid] = new_val
                    n_over += 1

            row["accepted"] = wid
            row["rows_overridden"] = n_over
            row["reason"] = ""
            if deltas:
                row["mean_abs_delta"] = float(np.mean(deltas))
                row["max_abs_delta"] = float(np.max(deltas))
            n_fired_total += 1
            n_rows_total += n_over

            log(f"[contact]  {wid}: known_rmse={rk:.4f}, n_contact={row['n_contact']}, "
                f"n_known={row['n_known']}, rows_overridden={n_over}")

        except Exception as e:
            row["reason"] = f"error: {e}"

        audit.append(row)

    log(f"[contact] contact override: {n_fired_total} wells fired, {n_rows_total} rows overridden")

    new_tvt = sub["id"].map(pred)
    assert new_tvt.notna().all(), "contact_override: NaN in output tvt"
    new_sub = sub.copy()
    new_sub["tvt"] = new_tvt.astype(float)

    audit_df = pd.DataFrame(audit, columns=[
        "well", "accepted", "paired_train", "rows_overridden", "reason",
        "n_contact", "n_known", "known_rmse", "mean_abs_delta", "max_abs_delta",
    ])
    return new_sub[["id", "tvt"]], audit_df

## Runner — the full pipeline in order

Executes everything defined above:

1. `main()` → base prediction and the shared PF cache.
2. **GRU leg** — every row the GRU covers is replaced by the pure GRU
   prediction; the base survives only on rows the leg could not produce.
3. **U-space projection** — per well, transform to `U = pred + z` anchored at
   the last known point, fit a robust degree-4 polynomial over normalized
   measured depth (IRLS with Cauchy weights on a MAD scale), and pull the
   prediction 75% toward the fit.
4. **Spatial outer blend** — W=0.30 toward the spatial-surface leg on guarded
   wells.
5. **Duplicate-well recovery**, then **contact override**.
6. Final sanity print of `submission.csv`.

In [ ]:
# ============================================================================
#  Runner: base engine -> GRU leg -> U-space projection -> spatial blend
#          -> duplicate-well recovery -> contact override
# ============================================================================
import pandas as _pd, numpy as _np

_test, _train = TEST, TRAIN

# spatial-leg configuration (frozen; equals spatial_leg.CONFIG)
SP_CONFIG = {'VARIANT': 'idw_trend', 'K': 20, 'SPW': 60, 'POWER': 2.0, 'B_CAL': 'late',
             'NN_DIST_MAX': 0.01, 'PREFIX_MIN': 30, 'PREFIX_RESID_MAX': 20.0,
             'ANISO_LAMBDA': 1.0, 'ANISO_MODE': 'pca'}

# ---- 1. base engine: features + GBDT-PP + 128-seed PF -> submission.csv ----
#         (also fills the shared PF cache the GRU leg reads)
main()
_m = _pd.read_csv("submission.csv")
print(f"[base] {len(_m)} rows, NaN={int(_m['tvt'].isna().sum())}", flush=True)

# ---- 2. GRU refinement leg (prediction body) -------------------------------
#         Every row the GRU covers is replaced by the pure GRU value; the base
#         prediction survives only on rows the leg could not produce.
try:
    _gw = _find_gru_weights()
    _tpaths = sorted(_test.glob('*__horizontal_well.csv'))
    _spfeat, _ = run_spatial_leg(_test, _train, SP_CONFIG, guards=False)  # sp_blate channel feed
    _gru_map = run_gru_leg(_tpaths, _test, _gw, GRU_DEPS, n_jobs=4, sp_map=_spfeat)
    _m = _pd.read_csv("submission.csv")
    _gv = _m["id"].map(_gru_map)
    _gok = _gv.notna()
    _m.loc[_gok, "tvt"] = _gv[_gok]
    _m[["id", "tvt"]].to_csv("submission.csv", index=False)
    print(f"[gru] applied {int(_gok.sum())}/{len(_m)} rows ({len(_gw)} models)", flush=True)
except Exception as _e_g:
    print(f"[gru] FAILED ({_e_g}), keeping the base prediction", flush=True)

# ---- 3. U-space projection --------------------------------------------------
#         Per well, U = pred + z anchored at the last known point should be a
#         smooth function of measured depth: fit a robust degree-4 polynomial
#         (IRLS, Cauchy weights, MAD scale) over normalized MD and pull the
#         prediction 75% toward the fit. Truth-free per-well post-process.
PROJ_DEG = 4
PROJ_LAM = 0.75
PROJ_IRLS = 4
PROJ_MIN_KNOWN = 5

def _proj_robfit(s, y, deg):
    """Robust polynomial fit: IRLS with Cauchy weights on a MAD scale."""
    if len(s) < deg + 2:
        return y.copy()
    c = _np.polyfit(s, y, deg)
    for _ in range(PROJ_IRLS):
        r = y - _np.polyval(c, s)
        sc = _np.median(_np.abs(r)) * 1.4826 + 1e-6
        c = _np.polyfit(s, y, deg, w=1.0 / (1.0 + (r / (2.0 * sc)) ** 2))
    return _np.polyval(c, s)

def _proj_well(pred, z, s, anchor, n_kn):
    if n_kn < PROJ_MIN_KNOWN:
        return pred
    u = (pred + z) - anchor
    fit = _proj_robfit(s, u, PROJ_DEG)
    full = (anchor + fit) - z
    out = (1.0 - PROJ_LAM) * pred + PROJ_LAM * full
    if not _np.all(_np.isfinite(out)):
        return pred
    return out

try:
    import time as _ptime
    _t_p0 = _ptime.time()
    _m = _pd.read_csv("submission.csv")
    _pos = {k: i for i, k in enumerate(_m["id"].tolist())}
    _tv = _m["tvt"].to_numpy("float64")
    _nw = 0; _nr = 0; _nskip = 0; _dsq = 0.0; _dmax = 0.0
    for _hp in sorted(_test.glob("*__horizontal_well.csv")):
        _wid = _hp.stem.replace("__horizontal_well", "")
        _hw = _pd.read_csv(_hp)
        _kn = _hw[_hw["TVT_input"].notna()]; _ev = _hw[_hw["TVT_input"].isna()]
        if len(_ev) == 0:
            continue
        if len(_kn) < PROJ_MIN_KNOWN:
            _nskip += 1; continue                      # guard: keep the base
        _ids = ["%s_%d" % (_wid, _i) for _i in _ev.index]
        if any(k not in _pos for k in _ids):
            _nskip += 1; continue
        _loc = _np.array([_pos[k] for k in _ids])
        _pred = _tv[_loc]
        if not _np.all(_np.isfinite(_pred)):
            _nskip += 1; continue
        _z = _ev["Z"].to_numpy(float)
        _mdv = _ev["MD"].to_numpy(float)
        _anchor = float(_kn["TVT_input"].iloc[-1]) + float(_kn["Z"].iloc[-1])
        _ps = float(_kn["MD"].iloc[-1]); _end = float(_hw["MD"].iloc[-1])
        _s = (_mdv - _ps) / max(_end - _ps, 1e-6)
        _out = _proj_well(_pred, _z, _s, _anchor, len(_kn))
        _d = _np.abs(_out - _pred)
        _tv[_loc] = _out
        _nw += 1; _nr += len(_loc)
        _dsq += float((_d ** 2).sum()); _dmax = max(_dmax, float(_d.max()))
    _m["tvt"] = _tv
    _m[["id", "tvt"]].to_csv("submission.csv", index=False)
    _drms = (_dsq / max(_nr, 1)) ** 0.5
    print("[proj] U-proj deg=%d lam=%.2f: applied %d wells / %d rows (skip %d) "
          "in %.1fs | |delta| rms=%.4f max=%.4f" % (
              PROJ_DEG, PROJ_LAM, _nw, _nr, _nskip, _ptime.time() - _t_p0,
              _drms, _dmax), flush=True)
except Exception as _e_pj:
    print("[proj] U-proj FAILED (%s), keeping pre-proj sub" % _e_pj, flush=True)

# ---- 4. spatial outer blend on guarded (tier-1) wells -----------------------
#         Wells with a well-constrained neighborhood get 30% of the
#         spatial-surface value (guards: PREFIX_MIN / NN_DIST_MAX /
#         PREFIX_RESID_MAX inside run_spatial_leg).
W_SP = 0.3
try:
    import time as _sptime
    _t_sp = _sptime.time()
    _sp_map, _sp_ok = run_spatial_leg(_test, _train, SP_CONFIG, guards=True)
    _m = _pd.read_csv("submission.csv")
    _spv = _m["id"].map(_sp_map)
    _spg = _m["id"].map(_sp_ok).fillna(False) & _spv.notna()
    _pre = _m.loc[_spg, "tvt"].to_numpy("float64")
    _m.loc[_spg, "tvt"] = (1.0 - W_SP) * _m.loc[_spg, "tvt"] + W_SP * _spv[_spg]
    _m[["id", "tvt"]].to_csv("submission.csv", index=False)
    _d2 = _np.abs(_m.loc[_spg, "tvt"].to_numpy("float64") - _pre)
    _d2rms = float(_np.sqrt(_np.mean(_d2 ** 2))) if len(_d2) else 0.0
    _d2max = float(_d2.max()) if len(_d2) else 0.0
    print(f"[spatial] tier-1 blend W={W_SP}: applied "
          f"{int(_spg.sum())}/{len(_m)} rows in {_sptime.time() - _t_sp:.1f}s | "
          f"|delta| rms={_d2rms:.4f} max={_d2max:.4f}", flush=True)
except Exception as _e_sp:
    print(f"[spatial] tier-1 blend FAILED ({_e_sp}), keeping proj sub", flush=True)

# ---- 5. duplicate-well recovery, then contact override ----------------------
_sub = _pd.read_csv("submission.csv")
_new_leg, _audit_leg = apply_dup_hedge(_sub, _test, _train, w_xfer=0.5)
_new_leg[["id", "tvt"]].to_csv("submission.csv", index=False)
_sub = _pd.read_csv("submission.csv")
_n_acc_leg = int((_audit_leg.get("accepted", _pd.Series([], dtype=object)).astype(str) != "").sum()) if len(_audit_leg) else 0
print(f"[dup] hedge: {_n_acc_leg} wells, {int(_audit_leg['rows_overridden'].sum()) if len(_audit_leg) else 0} rows", flush=True)
try:
    _new_co, _audit_co = apply_contact_override(_sub, _test, _train)
    _new_co[["id", "tvt"]].to_csv("submission.csv", index=False)
    _sub = _pd.read_csv("submission.csv")
    _n_acc_co = int((_audit_co["accepted"].astype(str) != "").sum()) if len(_audit_co) else 0
    print(f"[contact] override: {_n_acc_co} wells, {int(_audit_co['rows_overridden'].sum()) if len(_audit_co) else 0} rows", flush=True)
except Exception as _e_co:
    print(f"[contact] override FAILED ({_e_co}), keeping pre-contact sub", flush=True)

_chk = _pd.read_csv("submission.csv")
print(f"[final] {len(_chk)} rows, NaN={int(_chk['tvt'].isna().sum())}, "
      f"tvt[min/max/mean]={_chk['tvt'].min():.2f}/{_chk['tvt'].max():.2f}/{_chk['tvt'].mean():.2f}")